# Pipeline cache-first - Fingerprints

Este notebook roda em modo fingerprint-only por padrao. A comparacao/openSMILE fica desligada enquanto `use_cached_opensmile_results=False` e `rebuild_opensmile_block_cache=False`.

Objetivos desta rodada:

1. Nao reexecutar PyCaret, EDA ou selecao openSMILE ja concluida.
2. Carregar somente os datasets de fingerprints por bloco e raw peaks.
3. Avaliar cenarios de fingerprint com validacao por musica.
4. Exportar tabelas e graficos em `Dados/pycaret_fingerprint_outputs/fingerprint_vs_opensmile_refactor`.


## 1. Imports, configuracao e utilitarios


In [1]:
import re
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
from scipy.stats import pearsonr
from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error
from sklearn.model_selection import GroupKFold, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


@dataclass
class PipelineConfig:
    project_dir: Path = Path(r"C:\dev\python\TCC Ajustado")
    random_state: int = 42
    n_splits: int = 5
    n_estimators: int = 120
    n_jobs_model: int = -1
    n_jobs_cv: int = 1
    run_pycaret: bool = False
    use_cached_opensmile_results: bool = False
    rebuild_opensmile_block_cache: bool = False
    run_fingerprint_experiments: bool = True
    run_selectkbest_inside_cv: bool = True
    selector_k: int = 60
    model_mode: str = "compact_panel"
    include_ridge_reference: bool = True
    include_dummy_reference: bool = True
    export_html: bool = True
    export_png: bool = False
    show_figures: bool = True
    use_normalized_magnitude: bool = True

    song_id_col: str = "song_id"
    block_id_col: str = "block_id"
    start_col: str = "block_start_sec"
    end_col: str = "block_end_sec"
    time_col: str = "frameTime"
    targets: List[str] = field(default_factory=lambda: ["valence", "arousal"])

    data_dir: Path = field(init=False)
    raw_opensmile_dir: Path = field(init=False)
    baseline_output_dir: Path = field(init=False)
    fingerprint_output_dir: Path = field(init=False)
    fingerprint_block_path: Path = field(init=False)
    fingerprint_raw_peaks_path: Path = field(init=False)
    fingerprint_rank_block_path: Path = field(init=False)
    fingerprint_rank_raw_peaks_path: Path = field(init=False)
    experiment_dir: Path = field(init=False)
    tables_dir: Path = field(init=False)
    figures_dir: Path = field(init=False)
    cache_dir: Path = field(init=False)
    opensmile_block_cache_path: Path = field(init=False)
    raw_peaks_block_cache_path: Path = field(init=False)
    combined_block_cache_path: Path = field(init=False)

    def __post_init__(self) -> None:
        self.project_dir = Path(self.project_dir)
        self.data_dir = self.project_dir / "Dados"
        self.raw_opensmile_dir = self.data_dir / "features"
        self.baseline_output_dir = self.data_dir / "pycaret_baseline_outputs"
        self.fingerprint_output_dir = self.data_dir / "pycaret_fingerprint_outputs"
        self.fingerprint_block_path = self.data_dir / "fingerprint_band_rank" / "fingerprint_band_rank.parquet"
        self.fingerprint_raw_peaks_path = self.data_dir / "fingerprint_band_rank" / "fingerprint_band_rank_raw_peaks.parquet"
        self.fingerprint_rank_block_path = self.data_dir / "fingerprint_rank" / "fingerprint_rank.parquet"
        self.fingerprint_rank_raw_peaks_path = self.data_dir / "fingerprint_rank" / "fingerprint_rank_raw_peaks.parquet"
        self.experiment_dir = self.fingerprint_output_dir / "fingerprint_vs_opensmile_refactor"
        self.tables_dir = self.experiment_dir / "tables"
        self.figures_dir = self.experiment_dir / "figures"
        self.cache_dir = self.experiment_dir / "cache"
        self.opensmile_block_cache_path = self.cache_dir / "opensmile_block_cache.parquet"
        self.raw_peaks_block_cache_path = self.cache_dir / "raw_peaks_block_cache.parquet"
        self.combined_block_cache_path = self.cache_dir / "fingerprints_plus_opensmile_block_dataset.parquet"

    def ensure_dirs(self) -> None:
        for folder in [self.tables_dir, self.figures_dir, self.cache_dir]:
            folder.mkdir(parents=True, exist_ok=True)


META_COLS = {
    "song_id", "filename",
    "block_id", "block_start_sec", "block_end_sec", "block_duration_sec",
    "quadrant", "quadrant_label", "method", "title", "artist", "genre",
    # band-rank raw-peaks metadata columns
    "band_name", "band_id", "band_low_hz", "band_high_hz", "topk_per_band",
    # legacy / fallback names
    "block_idx", "start_time", "end_time", "duration_sec",
}


def progress_iter(iterable: Iterable, desc: Optional[str] = None, total: Optional[int] = None, leave: bool = True):
    return tqdm(iterable, desc=desc, total=total, dynamic_ncols=True, leave=leave)


def read_table(path: Path, **kwargs) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path, **kwargs)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path, **kwargs)

    attempts = [
        {"sep": ","},
        {"sep": ";"},
        {"sep": "\t"},
        {"sep": None, "engine": "python"},
    ]
    last_error = None
    for params in attempts:
        try:
            df_tmp = pd.read_csv(path, **{**params, **kwargs})
            if df_tmp.shape[1] > 1:
                df_tmp.columns = [str(c).strip() for c in df_tmp.columns]
                return df_tmp
        except Exception as exc:
            last_error = exc
    raise ValueError(f"Could not read {path}. Last error: {last_error}")


def save_table(cfg: PipelineConfig, df: pd.DataFrame, filename: str) -> Path:
    path = cfg.tables_dir / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved table: {path}")
    return path


def save_plot(cfg: PipelineConfig, fig, filename_stem: str) -> None:
    cfg.figures_dir.mkdir(parents=True, exist_ok=True)
    if cfg.export_html:
        fig.write_html(str(cfg.figures_dir / f"{filename_stem}.html"), include_plotlyjs="cdn")
    if cfg.export_png:
        try:
            fig.write_image(str(cfg.figures_dir / f"{filename_stem}.png"), scale=2)
        except Exception as exc:
            print(f"[WARN] PNG not exported for {filename_stem}: {exc}")


def pearson_safe(y_true, y_pred) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() < 3:
        return np.nan
    if np.nanstd(y_true[mask]) == 0 or np.nanstd(y_pred[mask]) == 0:
        return np.nan
    return float(pearsonr(y_true[mask], y_pred[mask])[0])


def rmse_metric(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


SCORING = {
    "rmse": make_scorer(rmse_metric, greater_is_better=False),
    "mae": make_scorer(mean_absolute_error, greater_is_better=False),
    "r2": "r2",
    "pearson": make_scorer(pearson_safe, greater_is_better=True),
}


## 2. Baseline openSMILE congelado


In [2]:
def load_opensmile_reference(cfg: PipelineConfig) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    all_results_path = cfg.baseline_output_dir / "all_model_results.csv"
    selected_features_path = cfg.baseline_output_dir / "selected_features_final.csv"
    missing_paths = [path for path in [all_results_path, selected_features_path] if not path.exists()]
    if missing_paths:
        print("[WARN] Baseline openSMILE/PyCaret nao encontrado; continuando sem baseline historico.")
        for path in missing_paths:
            print(f"       Arquivo ausente: {path}")
        print("       Para comparar contra openSMILE, coloque all_model_results.csv e selected_features_final.csv nessa pasta ou use use_cached_opensmile_results=False.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    all_results = read_table(all_results_path)
    selected_features = read_table(selected_features_path)

    for col in ["rmse_mean", "rmse_std", "mae_mean", "r2_mean"]:
        if col in all_results.columns:
            all_results[col] = pd.to_numeric(all_results[col], errors="coerce")

    status = all_results["status"] if "status" in all_results.columns else pd.Series("ok", index=all_results.index)
    ok = all_results[status.eq("ok")].copy()
    if ok.empty:
        print("[WARN] Baseline openSMILE carregado, mas sem linhas com status='ok'.")
        save_table(cfg, all_results, "opensmile_reference_all_model_results.csv")
        save_table(cfg, selected_features, "opensmile_reference_selected_features.csv")
        return all_results, pd.DataFrame(), selected_features
    best_idx = ok.sort_values(["target", "rmse_mean"]).groupby("target")["rmse_mean"].idxmin()
    best_by_target = ok.loc[best_idx].sort_values("target").reset_index(drop=True)
    best_by_target = best_by_target.assign(reference_source="cached_openSMILE_all_model_results")

    save_table(cfg, all_results, "opensmile_reference_all_model_results.csv")
    save_table(cfg, best_by_target, "opensmile_reference_best_by_target.csv")
    save_table(cfg, selected_features, "opensmile_reference_selected_features.csv")
    return all_results, best_by_target, selected_features


## 3. Carregamento dos fingerprints por bloco


In [3]:
def numeric_feature_columns(df: pd.DataFrame, targets: List[str], exclude: Optional[set] = None) -> List[str]:
    exclude = set(exclude or set())
    features = []
    for col in df.columns:
        if col in exclude or col in targets:
            continue
        converted = pd.to_numeric(df[col], errors="coerce")
        if converted.notna().sum() == 0:
            continue
        if converted.nunique(dropna=True) <= 1:
            continue
        features.append(col)
    return features


def load_fingerprint_block_table(cfg: PipelineConfig, path: Path, prefix: str) -> Tuple[pd.DataFrame, List[str]]:
    raw = read_table(path)
    raw.columns = [str(c).strip() for c in raw.columns]

    required = [cfg.song_id_col, cfg.block_id_col, cfg.start_col, cfg.end_col, *cfg.targets]
    missing = [c for c in required if c not in raw.columns]
    if missing:
        raise ValueError(f"{path.name} missing required columns: {missing}")

    raw[cfg.song_id_col] = pd.to_numeric(raw[cfg.song_id_col], errors="coerce").astype("Int64")
    raw[cfg.block_id_col] = pd.to_numeric(raw[cfg.block_id_col], errors="coerce").astype("Int64")
    raw[cfg.start_col] = pd.to_numeric(raw[cfg.start_col], errors="coerce")
    raw[cfg.end_col] = pd.to_numeric(raw[cfg.end_col], errors="coerce")
    for target in cfg.targets:
        raw[target] = pd.to_numeric(raw[target], errors="coerce")

    raw = raw.dropna(subset=[cfg.song_id_col, cfg.block_id_col, cfg.start_col, cfg.end_col]).copy()
    raw[cfg.song_id_col] = raw[cfg.song_id_col].astype(int)
    raw[cfg.block_id_col] = raw[cfg.block_id_col].astype(int)

    feature_cols_original = numeric_feature_columns(raw, targets=cfg.targets, exclude=META_COLS)
    if cfg.use_normalized_magnitude:
        # Preferir magnitude normalizada: remove col_db somente se col_norm_block existir
        _raw_col_set = set(raw.columns)

        def _has_norm_pair(c: str) -> bool:
            """True se a coluna c em dB tem versão normalizada no mesmo arquivo."""
            if c.endswith('_magnitude_db'):
                return (c[: -len('_magnitude_db')] + '_magnitude_norm_block') in _raw_col_set
            if c.endswith('_db') and '_mag_' in c:
                return (c[: -len('_db')] + '_norm_block') in _raw_col_set
            if 'energy_db_' in c:
                return c.replace('energy_db_', 'energy_norm_') in _raw_col_set
            if '_magnitude_db_' in c:
                return c.replace('_magnitude_db_', '_magnitude_norm_') in _raw_col_set
            if '_norm_' not in c and c.endswith('_magnitude_mean'):
                return (c + '_norm_block') in _raw_col_set
            return False

        _before = len(feature_cols_original)
        feature_cols_original = [c for c in feature_cols_original if not _has_norm_pair(c)]
        _removed = _before - len(feature_cols_original)
        if _removed:
            print(f"  [{path.name}] prefer_norm_magnitude: {_removed} features dB removidas (par norm disponivel)")
    rename_map = {col: f"{prefix}{col}" for col in feature_cols_original}
    keep_cols = [
        c
        for c in [cfg.song_id_col, cfg.block_id_col, cfg.start_col, cfg.end_col, "block_duration_sec", *cfg.targets]
        if c in raw.columns
    ]
    out = raw[keep_cols + feature_cols_original].rename(columns=rename_map)
    feature_cols = [rename_map[c] for c in feature_cols_original]
    for col in feature_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out, feature_cols


def load_raw_peaks_block_table(cfg: PipelineConfig, path: Path, prefix: str = "rawpeaks__") -> Tuple[pd.DataFrame, List[str]]:
    try:
        import pyarrow as pa
        import pyarrow.parquet as pq
    except Exception as exc:
        raise ImportError(f"pyarrow is required to stream {path.name}: {exc}")

    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    parquet_file = pq.ParquetFile(path)
    schema = parquet_file.schema_arrow

    required = [cfg.song_id_col, cfg.block_id_col, cfg.start_col, cfg.end_col, *cfg.targets]
    missing = [c for c in required if c not in schema.names]
    if missing:
        raise ValueError(f"{path.name} missing required columns: {missing}")

    excluded = set(META_COLS) | {cfg.song_id_col, cfg.block_id_col, cfg.start_col, cfg.end_col, *cfg.targets}
    numeric_feature_cols = [
        field.name
        for field in schema
        if field.name not in excluded and (
            pa.types.is_integer(field.type)
            or pa.types.is_floating(field.type)
            or pa.types.is_boolean(field.type)
        )
    ]

    if cfg.use_normalized_magnitude:
        numeric_feature_cols = [
            c for c in numeric_feature_cols
            if not (c.lower().endswith("_db") and "mag" in c.lower())
            and c.lower() != "magnitude"
        ]
    read_columns = [cfg.song_id_col, cfg.block_id_col, cfg.start_col, cfg.end_col, *cfg.targets, *numeric_feature_cols]
    aggregate_cols = list(dict.fromkeys([*cfg.targets, *numeric_feature_cols]))
    sum_accumulator: Optional[pd.DataFrame] = None
    count_accumulator: Optional[pd.DataFrame] = None

    for row_group_idx in range(parquet_file.num_row_groups):
        batch = parquet_file.read_row_group(row_group_idx, columns=read_columns).to_pandas()
        if batch.empty:
            continue

        for col in [cfg.song_id_col, cfg.block_id_col, cfg.start_col, cfg.end_col, *cfg.targets]:
            batch[col] = pd.to_numeric(batch[col], errors="coerce")

        batch = batch.dropna(subset=[cfg.song_id_col, cfg.block_id_col, cfg.start_col, cfg.end_col])
        if batch.empty:
            continue

        batch[cfg.song_id_col] = batch[cfg.song_id_col].astype(int)
        batch[cfg.block_id_col] = batch[cfg.block_id_col].astype(int)

        grouped = batch.groupby([cfg.song_id_col, cfg.block_id_col, cfg.start_col, cfg.end_col], sort=False)
        sum_part = grouped[aggregate_cols].sum(min_count=1)
        count_part = grouped[aggregate_cols].count()

        if sum_accumulator is None:
            sum_accumulator = sum_part
            count_accumulator = count_part
        else:
            sum_accumulator = sum_accumulator.add(sum_part, fill_value=0)
            count_accumulator = count_accumulator.add(count_part, fill_value=0)

    if sum_accumulator is None or count_accumulator is None or sum_accumulator.empty:
        raise ValueError(f"No rows were aggregated from {path.name}")

    out = sum_accumulator.divide(count_accumulator.replace(0, np.nan)).reset_index()
    out[cfg.song_id_col] = pd.to_numeric(out[cfg.song_id_col], errors="coerce").astype("Int64")
    out[cfg.block_id_col] = pd.to_numeric(out[cfg.block_id_col], errors="coerce").astype("Int64")
    out[cfg.song_id_col] = out[cfg.song_id_col].astype(int)
    out[cfg.block_id_col] = out[cfg.block_id_col].astype(int)
    for target in cfg.targets:
        out[target] = pd.to_numeric(out[target], errors="coerce")

    feature_cols = [f"{prefix}{col}" for col in numeric_feature_cols]
    out = out.rename(columns={col: f"{prefix}{col}" for col in numeric_feature_cols})
    for col in feature_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out, feature_cols


def load_fingerprint_dataset(cfg: PipelineConfig) -> Tuple[pd.DataFrame, Dict[str, List[str]], pd.DataFrame]:
    merge_keys = [cfg.song_id_col, cfg.block_id_col, cfg.start_col, cfg.end_col]

    band_block_df, band_block_features = load_fingerprint_block_table(cfg, cfg.fingerprint_block_path, "fpband__")
    rank_block_df, rank_block_features = load_fingerprint_block_table(cfg, cfg.fingerprint_rank_block_path, "fprank__")
    band_raw_df, band_raw_features = load_raw_peaks_block_table(cfg, cfg.fingerprint_raw_peaks_path, "rawpeaks_band__")
    rank_raw_df, rank_raw_features = load_raw_peaks_block_table(cfg, cfg.fingerprint_rank_raw_peaks_path, "rawpeaks_rank__")

    # Merge all tables onto the band_rank block table (which carries the targets)
    fp = band_block_df
    for other_df, other_features in [
        (rank_block_df, rank_block_features),
        (band_raw_df, band_raw_features),
        (rank_raw_df, rank_raw_features),
    ]:
        payload = other_df[merge_keys + other_features]
        fp = fp.merge(payload, on=merge_keys, how="inner")

    fp = fp.dropna(subset=cfg.targets).reset_index(drop=True)

    feature_sets = {
        "Band Rank (bloco)": band_block_features,
        "Rank (bloco)": rank_block_features,
        "Band Rank (raw peaks)": band_raw_features,
        "Rank (raw peaks)": rank_raw_features,
    }

    source_map = {
        "Band Rank (bloco)": cfg.fingerprint_block_path,
        "Rank (bloco)": cfg.fingerprint_rank_block_path,
        "Band Rank (raw peaks)": cfg.fingerprint_raw_peaks_path,
        "Rank (raw peaks)": cfg.fingerprint_rank_raw_peaks_path,
    }
    inventory = pd.DataFrame([
        {
            "feature_set": name,
            "source_path": str(source_map[name]),
            "n_features": len(features),
            "n_samples_with_targets": len(fp),
            "n_songs": fp[cfg.song_id_col].nunique(),
        }
        for name, features in feature_sets.items()
    ])
    save_table(cfg, inventory, "fingerprint_feature_inventory.csv")
    return fp, feature_sets, inventory


In [4]:
import re
from typing import Dict, List

def fingerprint_band_group(feature: str) -> str:
    raw = str(feature).replace("fpband__", "").replace("fprank__", "").replace("rawpeaks_band__", "").replace("rawpeaks_rank__", "").lower()
    if "very_high" in raw:
        return "very_high"
    if re.search(r"(^|_)high(_|$)", raw):
        return "high"
    if re.search(r"(^|_)mid(_|$)", raw):
        return "mid"
    if re.search(r"(^|_)low(_|$)", raw):
        return "low"
    if "top_" in raw or "top" in raw:
        return "top_rank"
    return "global"



def fingerprint_metric_group(feature: str) -> str:
    raw = str(feature).replace("fpband__", "").replace("fprank__", "").replace("rawpeaks_band__", "").replace("rawpeaks_rank__", "").lower()
    if "freq" in raw:
        return "frequency"
    if "mag" in raw or "magnitude" in raw:
        return "magnitude"
    if "rank" in raw:
        return "rank"
    if "midi" in raw or "semitone" in raw or "pitch" in raw or "octave" in raw:
        return "pitch_octave"
    if "count" in raw or raw.startswith("n_") or "event" in raw or "peak" in raw:
        return "counts"
    if "std" in raw:
        return "dispersion"
    if "mean" in raw:
        return "central_tendency"
    return "other"



def build_fingerprint_category_maps(features: List[str]) -> Dict[str, Dict[str, List[str]]]:
    unique_features = list(dict.fromkeys([str(feature) for feature in features]))
    bands = {
        band: [feature for feature in unique_features if fingerprint_band_group(feature) == band]
        for band in ["low", "mid", "high", "very_high"]
    }
    metric_groups = {
        metric: [feature for feature in unique_features if fingerprint_metric_group(feature) == metric]
        for metric in [
            "frequency",
            "magnitude",
            "rank",
            "pitch_octave",
            "counts",
            "dispersion",
            "central_tendency",
            "other",
        ]
    }
    return {"bands": bands, "metrics": metric_groups}



def exclude_band_from_features(features: List[str], band: str) -> List[str]:
    return [feature for feature in features if fingerprint_band_group(feature) != band]



def keep_band_only_features(features: List[str], band: str) -> List[str]:
    return [feature for feature in features if fingerprint_band_group(feature) == band]


## 4. Cache opcional de openSMILE por bloco


In [5]:
def extract_song_id_from_filename(path: Path) -> Optional[int]:
    match = re.search(r"(\d+)", path.stem)
    return int(match.group(1)) if match else None


def find_time_column(cfg: PipelineConfig, df: pd.DataFrame) -> str:
    candidates = [cfg.time_col, "frame_time", "time", "Time", "timestamp", "seconds", "sec"]
    for col in candidates:
        if col in df.columns:
            return col
    for col in df.columns:
        if "time" in str(col).lower():
            return col
    raise ValueError("No time column found in openSMILE features.")


def selected_opensmile_feature_union(selected_features: pd.DataFrame) -> List[str]:
    if selected_features.empty or "feature" not in selected_features.columns:
        return []
    return list(dict.fromkeys(selected_features["feature"].dropna().astype(str).tolist()))


def aggregate_song_opensmile_to_blocks(
    cfg: PipelineConfig,
    song_file: Path,
    blocks_for_song: pd.DataFrame,
    selected_features_union: List[str],
) -> pd.DataFrame:
    song_id = extract_song_id_from_filename(song_file)
    if song_id is None:
        return pd.DataFrame()

    raw = read_table(song_file)
    raw.columns = [str(c).strip() for c in raw.columns]
    time_col = find_time_column(cfg, raw)
    raw[time_col] = pd.to_numeric(raw[time_col], errors="coerce")
    raw = raw.dropna(subset=[time_col]).sort_values(time_col)

    existing_features = [c for c in selected_features_union if c in raw.columns]
    if not existing_features:
        return pd.DataFrame()

    for col in existing_features:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")

    rows = []
    for _, block in blocks_for_song.iterrows():
        start = float(block[cfg.start_col])
        end = float(block[cfg.end_col])
        window = raw[(raw[time_col] >= start) & (raw[time_col] < end)]
        if window.empty:
            continue
        row = {
            cfg.song_id_col: int(song_id),
            cfg.block_id_col: int(block[cfg.block_id_col]),
            cfg.start_col: start,
            cfg.end_col: end,
        }
        for col in existing_features:
            values = window[col]
            row[f"os__{col}__mean"] = float(values.mean())
            row[f"os__{col}__std"] = float(values.std(ddof=0))
        rows.append(row)
    return pd.DataFrame(rows)


def build_opensmile_block_cache(
    cfg: PipelineConfig,
    fingerprint_blocks: pd.DataFrame,
    selected_features: pd.DataFrame,
) -> pd.DataFrame:
    selected_union = selected_opensmile_feature_union(selected_features)
    if not selected_union:
        raise ValueError("No selected openSMILE features available for aggregation.")

    files = sorted(cfg.raw_opensmile_dir.glob("*.csv"))
    if not files:
        raise FileNotFoundError(cfg.raw_opensmile_dir)

    blocks_by_song = {
        int(song_id): grp.sort_values(cfg.start_col).reset_index(drop=True)
        for song_id, grp in fingerprint_blocks.groupby(cfg.song_id_col)
    }

    parts = []
    for song_file in progress_iter(files, desc="Aggregating openSMILE blocks", total=len(files)):
        song_id = extract_song_id_from_filename(song_file)
        blocks = blocks_by_song.get(song_id)
        if blocks is None or blocks.empty:
            continue
        try:
            part = aggregate_song_opensmile_to_blocks(cfg, song_file, blocks, selected_union)
            if not part.empty:
                parts.append(part)
        except Exception as exc:
            print(f"[WARN] Failed to aggregate {song_file.name}: {exc}")

    if not parts:
        raise ValueError("No openSMILE block rows were aggregated.")

    cache = pd.concat(parts, ignore_index=True)
    cache = cache.loc[:, ~cache.columns.duplicated()]
    cfg.opensmile_block_cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache.to_parquet(cfg.opensmile_block_cache_path, index=False)
    print(f"Saved openSMILE block cache: {cfg.opensmile_block_cache_path}")
    return cache


def load_or_build_opensmile_block_cache(
    cfg: PipelineConfig,
    fingerprint_df: pd.DataFrame,
    opensmile_selected_features: pd.DataFrame,
) -> Tuple[pd.DataFrame, List[str]]:
    if cfg.opensmile_block_cache_path.exists() and not cfg.rebuild_opensmile_block_cache:
        cache = pd.read_parquet(cfg.opensmile_block_cache_path)
        print(f"Loaded openSMILE block cache: {cfg.opensmile_block_cache_path}")
    elif cfg.rebuild_opensmile_block_cache:
        if opensmile_selected_features.empty:
            print("[WARN] Nao ha features openSMILE selecionadas para reconstruir o cache por bloco.")
            print("       Pulando cenarios openSMILE por bloco; os experimentos de fingerprint continuam.")
            return pd.DataFrame(), []
        cache = build_opensmile_block_cache(cfg, fingerprint_df, opensmile_selected_features)
    else:
        print("[INFO] openSMILE block cache not found. Combined openSMILE+fingerprint scenarios will be skipped.")
        print("       Set rebuild_opensmile_block_cache=True once to generate it from existing openSMILE CSV features.")
        return pd.DataFrame(), []

    feature_cols = [
        c
        for c in cache.columns
        if c not in {cfg.song_id_col, cfg.block_id_col, cfg.start_col, cfg.end_col}
        and pd.api.types.is_numeric_dtype(cache[c])
    ]
    return cache, feature_cols


## 5. Cenarios, modelos e avaliacao


In [6]:
def assemble_feature_scenarios(
    cfg: PipelineConfig,
    fingerprint_df: pd.DataFrame,
    fingerprint_feature_sets: Dict[str, List[str]],
    opensmile_block_cache: pd.DataFrame,
    opensmile_block_features: List[str],
) -> Tuple[pd.DataFrame, Dict[str, List[str]], pd.DataFrame]:
    dataset = fingerprint_df.copy()
    _block_source_type = {"Band Rank (bloco)", "Rank (bloco)"}
    scenarios = {
        name: list(dict.fromkeys(fingerprint_feature_sets.get(name, [])))
        for name in ["Band Rank (bloco)", "Rank (bloco)", "Band Rank (raw peaks)", "Rank (raw peaks)"]
    }

    scenario_rows = []
    for name, features in scenarios.items():
        scenario_rows.append(
            {
                "feature_set": name,
                "scenario_family": "fingerprint_source",
                "source_type": "block" if name in _block_source_type else "raw_peaks",
                "n_features": len(features),
                "n_samples": len(dataset),
                "n_songs": dataset[cfg.song_id_col].nunique(),
                "uses_opensmile_block_cache": False,
                "excluded_band": "",
                "metric_group": "",
                "band_group": "",
            }
        )

    scenario_inventory = pd.DataFrame(scenario_rows)
    save_table(cfg, scenario_inventory, "feature_scenario_inventory.csv")
    return dataset, scenarios, scenario_inventory


## 6. Comparacao, graficos, sintese e orquestracao


In [7]:
def compare_against_opensmile_reference(results: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:
    if results.empty or reference.empty:
        return pd.DataFrame()

    ref_cols = ["target", "model", "rmse_mean", "mae_mean", "r2_mean", "n_features", "cv"]
    ref = reference[[c for c in ref_cols if c in reference.columns]].copy()
    ref = ref.rename(
        columns={
            "model": "opensmile_reference_model",
            "rmse_mean": "opensmile_reference_rmse",
            "mae_mean": "opensmile_reference_mae",
            "r2_mean": "opensmile_reference_r2",
            "n_features": "opensmile_reference_n_features",
            "cv": "opensmile_reference_cv",
        }
    )

    comp = results[results["status"].eq("ok")].merge(ref, on="target", how="left")
    comp["delta_rmse_vs_openSMILE"] = comp["rmse_mean"] - comp["opensmile_reference_rmse"]
    comp["delta_rmse_pct_vs_openSMILE"] = 100.0 * comp["delta_rmse_vs_openSMILE"] / comp[
        "opensmile_reference_rmse"
    ]
    comp["impacto_vs_openSMILE"] = np.where(
        comp["delta_rmse_vs_openSMILE"] < 0,
        "melhorou",
        np.where(comp["delta_rmse_vs_openSMILE"] > 0, "piorou", "empatou"),
    )
    return comp.sort_values(["target", "delta_rmse_vs_openSMILE", "rmse_mean"]).reset_index(drop=True)


def best_result_tables(cfg: PipelineConfig, results: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if results.empty:
        best_by_feature_set = pd.DataFrame()
        best_new_by_target = pd.DataFrame()
    else:
        ok = results[results["status"].eq("ok")]
        best_by_feature_set = (
            ok.sort_values(["target", "feature_set", "rmse_mean"])
            .groupby(["target", "feature_set"], as_index=False)
            .first()
        )
        best_new_by_target = ok.sort_values(["target", "rmse_mean"]).groupby("target", as_index=False).first()
        dummy_by_target = (
            ok[ok["model"].eq("DummyMean")]
            .sort_values(["target", "rmse_mean"])
            .groupby("target", as_index=False)
            .first()[["target", "rmse_mean"]]
            .rename(columns={"rmse_mean": "dummy_rmse"})
        )
        best_by_feature_set = best_by_feature_set.merge(dummy_by_target, on="target", how="left")
        best_new_by_target = best_new_by_target.merge(dummy_by_target, on="target", how="left")
        best_by_feature_set["gain_vs_dummy_pct"] = 100.0 * (best_by_feature_set["dummy_rmse"] - best_by_feature_set["rmse_mean"]) / best_by_feature_set["dummy_rmse"]
        best_new_by_target["gain_vs_dummy_pct"] = 100.0 * (best_new_by_target["dummy_rmse"] - best_new_by_target["rmse_mean"]) / best_new_by_target["dummy_rmse"]

    save_table(cfg, best_by_feature_set, "best_result_by_target_and_feature_set.csv")
    save_table(cfg, best_new_by_target, "best_fingerprint_scenario_by_target.csv")
    return best_by_feature_set, best_new_by_target


def make_plots(
    cfg: PipelineConfig,
    best_by_feature_set: pd.DataFrame,
    comparison_vs_opensmile: pd.DataFrame,
) -> Dict[str, object]:
    figures: Dict[str, object] = {}
    if not best_by_feature_set.empty:
        feature_size_col = "n_features" if "n_features" in best_by_feature_set.columns else "n_features_effective"
        fig_rmse = px.bar(
            best_by_feature_set,
            x="feature_set",
            y="rmse_mean",
            color="target",
            barmode="group",
            text="rmse_mean",
            hover_data=["model", "selector", feature_size_col, "r2_mean", "pearson_mean", "cv"],
            title="Melhor RMSE por conjunto de features (GroupKFold por musica)",
        )
        fig_rmse.update_traces(texttemplate="%{text:.4f}", textposition="outside")
        fig_rmse.update_layout(template="plotly_white", xaxis_title="", yaxis_title="RMSE medio")
        save_plot(cfg, fig_rmse, "best_rmse_by_feature_set")
        if cfg.show_figures:
            fig_rmse.show()
        figures["best_rmse_by_feature_set"] = fig_rmse

    if not comparison_vs_opensmile.empty:
        plot_delta = comparison_vs_opensmile.copy()
        plot_delta["cenario"] = (
            plot_delta["feature_set"] + " | " + plot_delta["model"] + " | " + plot_delta["selector"]
        )
        fig_delta = px.bar(
            plot_delta,
            x="cenario",
            y="delta_rmse_vs_openSMILE",
            color="target",
            barmode="group",
            text="delta_rmse_vs_openSMILE",
            hover_data=[
                "rmse_mean",
                "opensmile_reference_rmse",
                "impacto_vs_openSMILE",
                "delta_rmse_pct_vs_openSMILE",
            ],
            title="Delta de RMSE contra o melhor baseline openSMILE salvo",
        )
        fig_delta.add_hline(y=0, line_dash="dash", line_color="black")
        fig_delta.update_traces(texttemplate="%{text:.4f}", textposition="outside")
        fig_delta.update_layout(template="plotly_white", xaxis_title="", yaxis_title="RMSE novo - RMSE openSMILE")
        save_plot(cfg, fig_delta, "delta_rmse_vs_cached_opensmile")
        if cfg.show_figures:
            fig_delta.show()
        figures["delta_rmse_vs_cached_opensmile"] = fig_delta
    return figures


def build_tcc_summary(cfg: PipelineConfig, opensmile_best_by_target: pd.DataFrame, best_new_by_target: pd.DataFrame, comparison_vs_opensmile: pd.DataFrame, opensmile_block_cache: pd.DataFrame) -> str:
    fingerprint_only = not cfg.use_cached_opensmile_results and not cfg.rebuild_opensmile_block_cache
    if fingerprint_only:
        lines = ["Pipeline em modo fingerprint-only: openSMILE foi pulado nesta rodada."]
    else:
        lines = ["Pipeline refatorado em modo cache-first: openSMILE permanece opcional e so entra quando habilitado na configuracao."]

    lines.append("Os cenarios avaliados usam apenas os conjuntos de features de fingerprint disponiveis no inventario.")

    if cfg.use_cached_opensmile_results and not opensmile_best_by_target.empty:
        for _, row in opensmile_best_by_target.iterrows():
            lines.append(
                f"Baseline openSMILE salvo para {row['target']}: {row['model']} "
                f"com RMSE={row['rmse_mean']:.4f}, R2={row['r2_mean']:.4f} e {int(row['n_features'])} features."
            )
    elif cfg.use_cached_opensmile_results:
        lines.append("Baseline openSMILE salvo nao foi encontrado para comparacao direta.")
    else:
        lines.append("Baseline e diagnosticos openSMILE nao foram carregados porque use_cached_opensmile_results=False.")

    comparison_cols = {"target", "feature_set", "model", "selector", "delta_rmse_vs_openSMILE", "impacto_vs_openSMILE"}
    has_opensmile_comparison = not comparison_vs_opensmile.empty and comparison_cols.issubset(comparison_vs_opensmile.columns)

    if best_new_by_target.empty:
        lines.append("Nenhum resultado novo de fingerprint foi gerado nesta execucao.")
    else:
        for _, row in best_new_by_target.iterrows():
            metric_parts = [f"RMSE={row['rmse_mean']:.4f}"]
            if "r2_mean" in row.index and pd.notna(row.get("r2_mean")):
                metric_parts.append(f"R2={row['r2_mean']:.4f}")
            if "gain_vs_dummy_pct" in row.index and pd.notna(row.get("gain_vs_dummy_pct")):
                metric_parts.append(f"ganho_vs_dummy={row['gain_vs_dummy_pct']:.1f}%")

            base_line = (
                f"Melhor cenario de fingerprint para {row['target']}: {row['feature_set']} "
                f"com {row['model']} ({row['selector']}), " + ", ".join(metric_parts)
            )

            if has_opensmile_comparison:
                comp = comparison_vs_opensmile[
                    (comparison_vs_opensmile["target"].eq(row["target"]))
                    & (comparison_vs_opensmile["feature_set"].eq(row["feature_set"]))
                    & (comparison_vs_opensmile["model"].eq(row["model"]))
                    & (comparison_vs_opensmile["selector"].eq(row["selector"]))
                ]
                if not comp.empty and pd.notna(comp.iloc[0].get("delta_rmse_vs_openSMILE")):
                    delta = comp.iloc[0]["delta_rmse_vs_openSMILE"]
                    impact = comp.iloc[0]["impacto_vs_openSMILE"]
                    base_line += f"; isso {impact} em {abs(delta):.4f} RMSE contra o baseline openSMILE salvo"

            lines.append(base_line + ".")

    if fingerprint_only:
        lines.append("Cache openSMILE por bloco nao foi carregado nem reconstruido nesta rodada.")
    elif opensmile_block_cache.empty:
        lines.append("O cache openSMILE por bloco nao foi usado nesta comparacao.")
    else:
        lines.append("O cache openSMILE por bloco foi carregado, mas os cenarios de fingerprint continuam separados no inventario.")

    summary_text = "\n".join(lines)
    summary_path = cfg.tables_dir / "fingerprint_vs_opensmile_summary_for_tcc.txt"
    summary_path.write_text(summary_text, encoding="utf-8")
    print(summary_text)
    print(f"\nSaved summary: {summary_path}")
    return summary_text


## 7. openSMILE detalhado: carregamento e comparacao ampla

Esta secao carrega todo o diretorio `pycaret_baseline_outputs`, nao apenas o ranking global. A partir desses CSVs ela monta uma tabela padronizada de referencias openSMILE para comparar contra os cenarios de fingerprints.

Sao considerados, quando existirem: rankings scikit-learn/GroupKFold, modelos nao lineares, split por segmento vs musica, seletores, sweep de `k`, anti-vazamento, antes/depois da selecao, dummy vs ridge, PyCaret compare/tuned, resultados por categoria Panda, correla??es, EDA e features selecionadas.


In [8]:
def _first_existing_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for col in candidates:
        if col in df.columns:
            return col
    return None


def _infer_target_from_table_name(table_name: str) -> Optional[str]:
    name = str(table_name).lower()
    if "valence" in name:
        return "valence"
    if "arousal" in name:
        return "arousal"
    return None


def _normalize_target(value, table_name: str = "") -> Optional[str]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return _infer_target_from_table_name(table_name)
    text = str(value).strip().lower()
    if text in {"valence", "arousal"}:
        return text
    return _infer_target_from_table_name(table_name)


def _row_value(row: pd.Series, *columns, default=np.nan):
    for col in columns:
        if col in row.index:
            return row[col]
    return default


def _numeric_or_nan(value) -> float:
    value = pd.to_numeric(pd.Series([value]), errors="coerce").iloc[0]
    return float(value) if pd.notna(value) else np.nan


def _append_metric_row(
    rows: List[Dict[str, object]],
    table_name: str,
    reference_family: str,
    row: pd.Series,
    target=None,
    reference_item=None,
    protocol=None,
    model=None,
    selector=None,
    rmse=None,
    rmse_std=None,
    mae=None,
    r2=None,
    n_features=None,
    n_samples=None,
    reference_status=None,
    notes="",
) -> None:
    target_norm = _normalize_target(target, table_name)
    rmse_value = _numeric_or_nan(rmse)
    if target_norm is None or pd.isna(rmse_value):
        return
    if reference_status is None:
        if str(table_name).startswith("pycaret_"):
            reference_status = "pycaret_suspect"
        elif table_name in {"split_comparison"}:
            reference_status = "incompatible_scale"
        else:
            reference_status = "valid_groupkfold"
    rows.append({
        "source_table": table_name,
        "reference_family": reference_family,
        "target": target_norm,
        "reference_item": str(reference_item) if reference_item is not None and pd.notna(reference_item) else reference_family,
        "protocol": str(protocol) if protocol is not None and pd.notna(protocol) else "",
        "model": str(model) if model is not None and pd.notna(model) else "",
        "selector": str(selector) if selector is not None and pd.notna(selector) else "",
        "rmse": rmse_value,
        "rmse_std": _numeric_or_nan(rmse_std),
        "mae": _numeric_or_nan(mae),
        "r2": _numeric_or_nan(r2),
        "n_features": _numeric_or_nan(n_features),
        "n_samples": _numeric_or_nan(n_samples),
        "reference_status": reference_status,
        "notes": notes,
    })


def _safe_loaded_table(loaded_tables: Dict[str, pd.DataFrame], name: str) -> pd.DataFrame:
    return loaded_tables.get(name, pd.DataFrame()).copy()


def load_all_opensmile_output_tables(cfg: PipelineConfig, baseline_dir: Optional[Path] = None) -> Dict[str, object]:
    baseline_dir = Path(baseline_dir or cfg.baseline_output_dir)
    loaded_tables: Dict[str, pd.DataFrame] = {}
    manifest_rows = []

    for path in sorted(baseline_dir.glob("*.csv")):
        try:
            df_table = read_table(path)
            loaded_tables[path.stem] = df_table
            manifest_rows.append({
                "table": path.stem,
                "file": path.name,
                "rows": int(df_table.shape[0]),
                "columns": int(df_table.shape[1]),
                "column_list": ", ".join(map(str, df_table.columns[:20])),
                "loaded": True,
                "error": "",
            })
        except Exception as exc:
            manifest_rows.append({
                "table": path.stem,
                "file": path.name,
                "rows": 0,
                "columns": 0,
                "column_list": "",
                "loaded": False,
                "error": str(exc),
            })

    eda_summary_path = baseline_dir / "eda_summary_for_tcc.md"
    eda_summary_text = eda_summary_path.read_text(encoding="utf-8") if eda_summary_path.exists() else ""

    manifest = pd.DataFrame(manifest_rows).sort_values(["loaded", "table"], ascending=[False, True]).reset_index(drop=True)
    save_table(cfg, manifest, "opensmile_detailed_loaded_tables_manifest.csv")
    return {
        "baseline_dir": baseline_dir,
        "loaded_tables": loaded_tables,
        "manifest": manifest,
        "eda_summary_text": eda_summary_text,
    }


def build_opensmile_metric_reference_long(cfg: PipelineConfig, loaded_tables: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []

    for table_name in ["all_model_results", "baseline_sklearn_valence", "baseline_sklearn_arousal"]:
        df_table = _safe_loaded_table(loaded_tables, table_name)
        for _, row in df_table.iterrows():
            _append_metric_row(
                rows, table_name, "openSMILE_model_ranking_groupkfold", row,
                target=_row_value(row, "target", "Target"),
                reference_item=_row_value(row, "model", "Model"),
                protocol=_row_value(row, "cv"),
                model=_row_value(row, "model", "Model"),
                rmse=_row_value(row, "rmse_mean", "RMSE", "rmse"),
                rmse_std=_row_value(row, "rmse_std", "RMSE (std)"),
                mae=_row_value(row, "mae_mean", "MAE", "mae"),
                r2=_row_value(row, "r2_mean", "R2", "r2"),
                n_features=_row_value(row, "n_features", "N features"),
                n_samples=_row_value(row, "n_samples"),
            )

    df_table = _safe_loaded_table(loaded_tables, "nonlinear_grouped_results")
    for _, row in df_table.iterrows():
        _append_metric_row(
            rows, "nonlinear_grouped_results", "openSMILE_nonlinear_groupkfold", row,
            target=_row_value(row, "target"),
            reference_item=_row_value(row, "model"),
            protocol=_row_value(row, "cv"),
            model=_row_value(row, "model"),
            rmse=_row_value(row, "rmse"),
            rmse_std=_row_value(row, "rmse_std"),
            mae=_row_value(row, "mae"),
            r2=_row_value(row, "r2"),
        )

    df_table = _safe_loaded_table(loaded_tables, "split_comparison")
    for _, row in df_table.iterrows():
        _append_metric_row(
            rows, "split_comparison", "openSMILE_split_segmento_vs_musica", row,
            target=_row_value(row, "target"),
            reference_item=f"{_row_value(row, 'split')} | {_row_value(row, 'model')}",
            protocol=_row_value(row, "split"),
            model=_row_value(row, "model"),
            rmse=_row_value(row, "rmse"),
            rmse_std=_row_value(row, "rmse_std"),
            mae=_row_value(row, "mae"),
            r2=_row_value(row, "r2"),
            notes="Segment KFold is diagnostic and can be optimistic versus GroupKFold.",
        )

    df_table = _safe_loaded_table(loaded_tables, "selection_methods_comparison")
    for _, row in df_table.iterrows():
        _append_metric_row(
            rows, "selection_methods_comparison", "openSMILE_feature_selection_methods", row,
            target=_row_value(row, "target"),
            reference_item=_row_value(row, "selector"),
            protocol="GroupKFold / Ridge",
            model="Ridge",
            selector=_row_value(row, "selector"),
            rmse=_row_value(row, "rmse"),
            rmse_std=_row_value(row, "rmse_std"),
            r2=_row_value(row, "r2"),
        )

    df_table = _safe_loaded_table(loaded_tables, "selector_k_sweep_grouped")
    for _, row in df_table.iterrows():
        k = _row_value(row, "k")
        _append_metric_row(
            rows, "selector_k_sweep_grouped", "openSMILE_selectkbest_k_sweep_groupkfold", row,
            target=_row_value(row, "target"),
            reference_item=f"SelectKBest k={k}",
            protocol="GroupKFold / Ridge",
            model="Ridge",
            selector=f"SelectKBest k={k}",
            rmse=_row_value(row, "rmse"),
            r2=_row_value(row, "r2"),
        )

    df_table = _safe_loaded_table(loaded_tables, "selector_k_sweep_results")
    for _, row in df_table.iterrows():
        k = _row_value(row, "k_effective", "k_requested")
        protocol = f"{_row_value(row, 'cenario_validacao')} | {_row_value(row, 'cv')}"
        _append_metric_row(
            rows, "selector_k_sweep_results", "openSMILE_selectkbest_k_sweep_full", row,
            target=_row_value(row, "target"),
            reference_item=f"{_row_value(row, 'selector')} k={k}",
            protocol=protocol,
            model="Ridge",
            selector=f"{_row_value(row, 'selector')} k={k}",
            rmse=_row_value(row, "rmse_mean"),
            rmse_std=_row_value(row, "rmse_std"),
            mae=_row_value(row, "mae_mean"),
            r2=_row_value(row, "r2_mean"),
            n_features=_row_value(row, "n_features"),
        )

    df_table = _safe_loaded_table(loaded_tables, "antileakage_pipeline_results")
    for _, row in df_table.iterrows():
        _append_metric_row(
            rows, "antileakage_pipeline_results", "openSMILE_antileakage_pipeline", row,
            target=_row_value(row, "Target"),
            reference_item=_row_value(row, "Pipeline"),
            protocol="5-fold CV",
            model="Ridge",
            selector=_row_value(row, "Pipeline"),
            rmse=_row_value(row, "RMSE (media)"),
            rmse_std=_row_value(row, "RMSE (std)"),
            r2=_row_value(row, "R2 (media)"),
        )

    df_table = _safe_loaded_table(loaded_tables, "before_after_selection")
    for _, row in df_table.iterrows():
        _append_metric_row(
            rows, "before_after_selection", "openSMILE_before_after_selection", row,
            target=_row_value(row, "Target"),
            reference_item=_row_value(row, "Conjunto"),
            protocol="Ridge / 5-fold CV",
            model="Ridge",
            selector=_row_value(row, "Conjunto"),
            rmse=_row_value(row, "RMSE"),
            r2=_row_value(row, "R2"),
            n_features=_row_value(row, "N features"),
        )

    df_table = _safe_loaded_table(loaded_tables, "dummy_vs_ridge_baseline")
    for _, row in df_table.iterrows():
        _append_metric_row(
            rows, "dummy_vs_ridge_baseline", "openSMILE_dummy_vs_ridge", row,
            target=_row_value(row, "Target"),
            reference_item=_row_value(row, "Modelo"),
            protocol="Baseline diagnostic",
            model=_row_value(row, "Modelo"),
            rmse=_row_value(row, "RMSE"),
            r2=_row_value(row, "R2"),
        )

    df_table = _safe_loaded_table(loaded_tables, "per_category_model_results")
    for _, row in df_table.iterrows():
        _append_metric_row(
            rows, "per_category_model_results", "openSMILE_panda_category_ridge", row,
            target=_row_value(row, "Target"),
            reference_item=_row_value(row, "Categoria Panda"),
            protocol="Ridge / GroupKFold",
            model="Ridge",
            selector="Categoria Panda",
            rmse=_row_value(row, "RMSE"),
            r2=_row_value(row, "R2"),
            n_features=_row_value(row, "N features"),
        )

    for table_name in [
        "pycaret_compare_valence", "pycaret_compare_arousal",
        "pycaret_compare_valence_DEAM_openSMILE_valence", "pycaret_compare_arousal_DEAM_openSMILE_arousal",
    ]:
        df_table = _safe_loaded_table(loaded_tables, table_name)
        for _, row in df_table.iterrows():
            _append_metric_row(
                rows, table_name, "openSMILE_pycaret_compare", row,
                target=_infer_target_from_table_name(table_name),
                reference_item=_row_value(row, "Model"),
                protocol="PyCaret internal CV",
                model=_row_value(row, "Model"),
                rmse=_row_value(row, "RMSE"),
                mae=_row_value(row, "MAE"),
                r2=_row_value(row, "R2"),
                notes="PyCaret results may use a different validation setup; compare as diagnostic context.",
            )

    for table_name in [
        "pycaret_tuned_valence", "pycaret_tuned_arousal",
        "pycaret_tuned_valence_DEAM_openSMILE_valence", "pycaret_tuned_arousal_DEAM_openSMILE_arousal",
    ]:
        df_table = _safe_loaded_table(loaded_tables, table_name)
        for idx, row in df_table.iterrows():
            _append_metric_row(
                rows, table_name, "openSMILE_pycaret_tuned", row,
                target=_infer_target_from_table_name(table_name),
                reference_item=f"tuned_row_{idx}",
                protocol="PyCaret tuned CV row",
                model="PyCaret tuned best",
                rmse=_row_value(row, "RMSE"),
                mae=_row_value(row, "MAE"),
                r2=_row_value(row, "R2"),
                notes="PyCaret tuned output is kept as detailed context, not the primary GroupKFold reference.",
            )

    metric_reference = pd.DataFrame(rows)
    if not metric_reference.empty:
        metric_reference = metric_reference.sort_values(["target", "rmse", "reference_family", "reference_item"]).reset_index(drop=True)
    save_table(cfg, metric_reference, "opensmile_metric_reference_long.csv")
    return metric_reference


def build_opensmile_feature_diagnostics(cfg: PipelineConfig, loaded_tables: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    selected_features = _safe_loaded_table(loaded_tables, "selected_features_final")
    selected_by_category = pd.DataFrame()
    if not selected_features.empty:
        selected_by_category = (
            selected_features
            .groupby(["target", "categoria_musical", "method"], dropna=False)
            .agg(
                n_selected=("feature", "nunique"),
                first_selected_order=("selected_order", "min"),
                feature_examples=("feature", lambda s: ", ".join(s.astype(str).head(6))),
            )
            .reset_index()
            .sort_values(["target", "n_selected"], ascending=[True, False])
        )
        save_table(cfg, selected_by_category, "opensmile_selected_features_by_category_detailed.csv")

    corr_frames = []
    for table_name in ["corr_features_valence", "corr_features_arousal"]:
        df_corr = _safe_loaded_table(loaded_tables, table_name)
        if not df_corr.empty:
            corr_frames.append(df_corr.assign(source_table=table_name))
    top_correlations = pd.DataFrame()
    if corr_frames:
        top_correlations = pd.concat(corr_frames, ignore_index=True)
        if "abs_pearson" in top_correlations.columns:
            top_correlations = (
                top_correlations
                .sort_values(["target", "abs_pearson"], ascending=[True, False])
                .groupby("target", as_index=False)
                .head(20)
                .reset_index(drop=True)
            )
            save_table(cfg, top_correlations, "opensmile_top20_correlations_by_target.csv")

    eda_structure = _safe_loaded_table(loaded_tables, "eda_data_structure")
    outlier_report = _safe_loaded_table(loaded_tables, "eda_outlier_report_iqr")
    category_summary = _safe_loaded_table(loaded_tables, "feature_category_summary")
    eda_compact = pd.DataFrame()
    if not eda_structure.empty:
        eda_compact = pd.DataFrame([{
            "n_features_eda": int(eda_structure.shape[0]),
            "mean_missing_rate_pct": _numeric_or_nan(eda_structure.get("missing_rate_%", pd.Series(dtype=float)).mean()),
            "max_missing_rate_pct": _numeric_or_nan(eda_structure.get("missing_rate_%", pd.Series(dtype=float)).max()),
            "n_recommended_keep": int(eda_structure.get("recommendation", pd.Series(dtype=str)).astype(str).str.contains("Manter", case=False, na=False).sum()),
            "n_outlier_features": int(outlier_report.shape[0]) if not outlier_report.empty else np.nan,
            "max_outlier_rate_pct": _numeric_or_nan(outlier_report.get("outlier_rate_%", pd.Series(dtype=float)).max()) if not outlier_report.empty else np.nan,
            "n_panda_categories": int(category_summary.shape[0]) if not category_summary.empty else np.nan,
        }])
        save_table(cfg, eda_compact, "opensmile_eda_compact_summary.csv")

    return {
        "selected_by_category": selected_by_category,
        "top_correlations": top_correlations,
        "eda_compact_summary": eda_compact,
        "feature_category_summary": category_summary,
    }


def load_detailed_opensmile_outputs(cfg: PipelineConfig) -> Dict[str, object]:
    loaded = load_all_opensmile_output_tables(cfg)
    metric_reference = build_opensmile_metric_reference_long(cfg, loaded["loaded_tables"])
    diagnostics = build_opensmile_feature_diagnostics(cfg, loaded["loaded_tables"])

    best_by_family = pd.DataFrame()
    best_global = pd.DataFrame()
    if not metric_reference.empty:
        refs_ok = metric_reference.dropna(subset=["rmse"]).copy()
        valid_refs = refs_ok[refs_ok["reference_status"].eq("valid_groupkfold")].copy()
        status_summary = (
            metric_reference.groupby(["reference_family", "reference_status"], dropna=False)
            .size()
            .reset_index(name="n_rows")
            .sort_values(["reference_status", "reference_family"])
        )
        save_table(cfg, status_summary, "opensmile_reference_status_summary.csv")
        best_by_family = (
            valid_refs
            .sort_values(["target", "reference_family", "rmse"])
            .groupby(["target", "reference_family"], as_index=False)
            .first()
        )
        best_global = (
            valid_refs
            .sort_values(["target", "rmse"])
            .groupby("target", as_index=False)
            .first()
        )
        save_table(cfg, best_by_family, "opensmile_best_reference_by_family.csv")
        save_table(cfg, best_global, "opensmile_best_reference_global_detailed.csv")

    return {
        **loaded,
        "metric_reference_long": metric_reference,
        "best_by_family": best_by_family,
        "best_global": best_global,
        **diagnostics,
    }


def _fingerprint_rows_for_comparison(results: pd.DataFrame, best_only: bool = True) -> pd.DataFrame:
    if results.empty:
        return pd.DataFrame()
    fp = results[results["status"].eq("ok")].copy()
    if best_only:
        fp = (
            fp.sort_values(["target", "feature_set", "rmse_mean"])
            .groupby(["target", "feature_set"], as_index=False)
            .first()
        )
    rename_map = {
        "feature_set": "fingerprint_feature_set",
        "model": "fingerprint_model",
        "selector": "fingerprint_selector",
        "cv": "fingerprint_cv",
        "n_samples": "fingerprint_n_samples",
        "n_songs": "fingerprint_n_songs",
        "n_features": "fingerprint_n_features",
        "rmse_mean": "fingerprint_rmse",
        "rmse_std": "fingerprint_rmse_std",
        "mae_mean": "fingerprint_mae",
        "r2_mean": "fingerprint_r2",
        "pearson_mean": "fingerprint_pearson",
    }
    keep_cols = [c for c in ["target", *rename_map.keys()] if c in fp.columns]
    return fp[keep_cols].rename(columns=rename_map).reset_index(drop=True)


def _opensmile_rows_for_comparison(metric_reference: pd.DataFrame, best_by_family: bool = False) -> pd.DataFrame:
    if metric_reference.empty:
        return pd.DataFrame()
    refs = metric_reference.dropna(subset=["rmse"]).copy()
    if best_by_family:
        refs = (
            refs.sort_values(["target", "reference_family", "rmse"])
            .groupby(["target", "reference_family"], as_index=False)
            .first()
        )
    rename_map = {
        "source_table": "opensmile_source_table",
        "reference_family": "opensmile_reference_family",
        "reference_item": "opensmile_reference_item",
        "protocol": "opensmile_protocol",
        "model": "opensmile_model",
        "selector": "opensmile_selector",
        "rmse": "opensmile_rmse",
        "rmse_std": "opensmile_rmse_std",
        "mae": "opensmile_mae",
        "r2": "opensmile_r2",
        "n_features": "opensmile_n_features",
        "n_samples": "opensmile_n_samples",
        "notes": "opensmile_notes",
    }
    keep_cols = [c for c in ["target", *rename_map.keys()] if c in refs.columns]
    return refs[keep_cols].rename(columns=rename_map).reset_index(drop=True)


def _add_opensmile_delta_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    out = df.copy()
    out["delta_rmse_fingerprint_minus_openSMILE"] = out["fingerprint_rmse"] - out["opensmile_rmse"]
    out["delta_rmse_pct_fingerprint_minus_openSMILE"] = 100.0 * out["delta_rmse_fingerprint_minus_openSMILE"] / out["opensmile_rmse"]
    out["impacto_fingerprint_vs_openSMILE"] = np.where(
        out["delta_rmse_fingerprint_minus_openSMILE"] < 0,
        "fingerprint_melhorou",
        np.where(out["delta_rmse_fingerprint_minus_openSMILE"] > 0, "fingerprint_piorou", "empate"),
    )
    return out.sort_values(["target", "delta_rmse_fingerprint_minus_openSMILE", "fingerprint_feature_set"]).reset_index(drop=True)


def compare_fingerprint_results_with_detailed_opensmile(
    cfg: PipelineConfig,
    fingerprint_results: pd.DataFrame,
    opensmile_detail: Dict[str, object],
) -> Dict[str, pd.DataFrame]:
    metric_reference = opensmile_detail.get("metric_reference_long", pd.DataFrame())

    fp_best_by_feature_set = _fingerprint_rows_for_comparison(fingerprint_results, best_only=True)
    fp_all_ok = _fingerprint_rows_for_comparison(fingerprint_results, best_only=False)
    os_all_refs = _opensmile_rows_for_comparison(metric_reference, best_by_family=False)
    os_best_family = _opensmile_rows_for_comparison(metric_reference, best_by_family=True)

    best_feature_set_vs_best_family = pd.DataFrame()
    best_feature_set_vs_all_refs = pd.DataFrame()
    all_fingerprint_vs_best_family = pd.DataFrame()

    if not fp_best_by_feature_set.empty and not os_best_family.empty:
        best_feature_set_vs_best_family = _add_opensmile_delta_columns(
            fp_best_by_feature_set.merge(os_best_family, on="target", how="inner")
        )
        save_table(cfg, best_feature_set_vs_best_family, "fingerprint_best_feature_sets_vs_opensmile_best_by_family.csv")

    if not fp_best_by_feature_set.empty and not os_all_refs.empty:
        best_feature_set_vs_all_refs = _add_opensmile_delta_columns(
            fp_best_by_feature_set.merge(os_all_refs, on="target", how="inner")
        )
        save_table(cfg, best_feature_set_vs_all_refs, "fingerprint_best_feature_sets_vs_all_opensmile_references.csv")

    if not fp_all_ok.empty and not os_best_family.empty:
        all_fingerprint_vs_best_family = _add_opensmile_delta_columns(
            fp_all_ok.merge(os_best_family, on="target", how="inner")
        )
        save_table(cfg, all_fingerprint_vs_best_family, "fingerprint_all_rows_vs_opensmile_best_by_family.csv")

    return {
        "fingerprint_best_by_feature_set": fp_best_by_feature_set,
        "opensmile_all_references_for_comparison": os_all_refs,
        "opensmile_best_by_family_for_comparison": os_best_family,
        "best_feature_set_vs_best_family": best_feature_set_vs_best_family,
        "best_feature_set_vs_all_refs": best_feature_set_vs_all_refs,
        "all_fingerprint_vs_best_family": all_fingerprint_vs_best_family,
    }


def make_detailed_opensmile_comparison_plots(
    cfg: PipelineConfig,
    opensmile_detail: Dict[str, object],
    detailed_comparison: Dict[str, pd.DataFrame],
) -> Dict[str, object]:
    figures = {}
    best_by_family = opensmile_detail.get("best_by_family", pd.DataFrame())
    if not best_by_family.empty:
        fig = px.bar(
            best_by_family,
            x="reference_family",
            y="rmse",
            color="target",
            barmode="group",
            text="rmse",
            hover_data=["reference_item", "model", "selector", "protocol", "r2", "n_features", "source_table"],
            title="openSMILE: melhor RMSE por familia de referencia carregada",
        )
        fig.update_traces(texttemplate="%{text:.4f}", textposition="outside")
        fig.update_layout(template="plotly_white", xaxis_title="", yaxis_title="RMSE")
        save_plot(cfg, fig, "opensmile_best_rmse_by_reference_family")
        if cfg.show_figures:
            fig.show()
        figures["opensmile_best_rmse_by_reference_family"] = fig

    family_comp = detailed_comparison.get("best_feature_set_vs_best_family", pd.DataFrame())
    if not family_comp.empty:
        fig = px.bar(
            family_comp,
            x="opensmile_reference_family",
            y="delta_rmse_fingerprint_minus_openSMILE",
            color="fingerprint_feature_set",
            facet_col="target",
            barmode="group",
            text="delta_rmse_fingerprint_minus_openSMILE",
            hover_data=["fingerprint_model", "fingerprint_selector", "fingerprint_rmse", "opensmile_reference_item", "opensmile_rmse", "impacto_fingerprint_vs_openSMILE"],
            title="Fingerprints vs openSMILE: delta de RMSE contra cada familia de referencia",
        )
        fig.add_hline(y=0, line_dash="dash", line_color="black")
        fig.update_traces(texttemplate="%{text:.4f}", textposition="outside")
        fig.update_layout(template="plotly_white", xaxis_title="", yaxis_title="RMSE fingerprint - RMSE openSMILE")
        save_plot(cfg, fig, "fingerprint_delta_vs_opensmile_reference_families")
        if cfg.show_figures:
            fig.show()
        figures["fingerprint_delta_vs_opensmile_reference_families"] = fig

    return figures


## 8. Parametros desta execucao

O modo atual para `Run All` fica em fingerprint-only:

- `run_pycaret=False`: nao roda PyCaret novamente.
- `use_cached_opensmile_results=False`: nao carrega baseline, tabelas ou diagnosticos openSMILE.
- `rebuild_opensmile_block_cache=False`: nao reler CSVs openSMILE nem reconstruir cache por bloco.
- `model_mode="compact_panel"`: avalia um painel compacto de modelos nos cenarios de fingerprint.

Quando quiser voltar a comparar com openSMILE, use a secao final do notebook para reabilitar os flags.


In [9]:
cfg = PipelineConfig(
    project_dir=Path(r"C:\dev\python\TCC Ajustado"),
    run_pycaret=False,
    use_cached_opensmile_results=False,
    rebuild_opensmile_block_cache=False,
    run_fingerprint_experiments=True,
    run_selectkbest_inside_cv=True,
    selector_k=60,
    model_mode="compact_panel",
    include_ridge_reference=True,
    include_dummy_reference=True,
    n_estimators=120,
    n_jobs_model=-1,
    n_jobs_cv=1,
    export_html=True,
    export_png=False,
    show_figures=True,
)

cfg.ensure_dirs()
print(f"Projeto: {cfg.project_dir}")
print(f"Saidas: {cfg.experiment_dir}")
print("Modo: fingerprint-only (openSMILE pulado nesta rodada)")
print(f"Fingerprint blocos: {cfg.fingerprint_block_path}")
print(f"Fingerprint raw_peaks: {cfg.fingerprint_raw_peaks_path}")


Projeto: C:\dev\python\TCC Ajustado
Saidas: C:\dev\python\TCC Ajustado\Dados\pycaret_fingerprint_outputs\fingerprint_vs_opensmile_refactor
Modo: fingerprint-only (openSMILE pulado nesta rodada)
Fingerprint blocos: C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank\fingerprint_band_rank.parquet
Fingerprint raw_peaks: C:\dev\python\TCC Ajustado\Dados\fingerprint_band_rank\fingerprint_band_rank_raw_peaks.parquet


## 9. Executar pipeline refatorado


In [10]:
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from scipy.stats import ttest_rel, wilcoxon
except Exception:
    ttest_rel = None
    wilcoxon = None


def compare_against_opensmile_reference(results: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:
    if results.empty or reference.empty:
        return pd.DataFrame()

    ref_cols = ["target", "model", "rmse_mean", "mae_mean", "r2_mean", "n_features", "cv"]
    ref = reference[[c for c in ref_cols if c in reference.columns]].copy()
    ref = ref.rename(columns={
        "model": "opensmile_reference_model",
        "rmse_mean": "opensmile_reference_rmse",
        "mae_mean": "opensmile_reference_mae",
        "r2_mean": "opensmile_reference_r2",
        "n_features": "opensmile_reference_n_features",
        "cv": "opensmile_reference_cv",
    })

    comp = results[results["status"].eq("ok")].merge(ref, on="target", how="left")
    comp["delta_rmse_vs_openSMILE"] = comp["rmse_mean"] - comp["opensmile_reference_rmse"]
    comp["delta_rmse_pct_vs_openSMILE"] = 100.0 * comp["delta_rmse_vs_openSMILE"] / comp["opensmile_reference_rmse"]
    comp["impacto_vs_openSMILE"] = np.where(comp["delta_rmse_vs_openSMILE"] < 0, "melhorou", np.where(comp["delta_rmse_vs_openSMILE"] > 0, "piorou", "empatou"))
    return comp.sort_values(["target", "delta_rmse_vs_openSMILE", "rmse_mean"]).reset_index(drop=True)


def model_factory(cfg: PipelineConfig, model_name: str):
    factories = {
        "DummyMean": lambda: (DummyRegressor(strategy="mean"), False),
        "Ridge": lambda: (Ridge(alpha=1.0, random_state=cfg.random_state), True),
        "LinearRegression": lambda: (LinearRegression(), True),
        "Lasso": lambda: (Lasso(alpha=0.001, random_state=cfg.random_state, max_iter=5000), True),
        "ElasticNet": lambda: (ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=cfg.random_state, max_iter=5000), True),
        "ExtraTrees": lambda: (ExtraTreesRegressor(n_estimators=cfg.n_estimators, min_samples_leaf=2, random_state=cfg.random_state, n_jobs=cfg.n_jobs_model), False),
        "RandomForest": lambda: (RandomForestRegressor(n_estimators=cfg.n_estimators, min_samples_leaf=2, random_state=cfg.random_state, n_jobs=cfg.n_jobs_model), False),
        "GradientBoosting": lambda: (GradientBoostingRegressor(n_estimators=cfg.n_estimators, random_state=cfg.random_state), False),
    }
    if model_name not in factories:
        model_name = "Ridge"
    return factories[model_name]()


def model_names_for_target(cfg: PipelineConfig, target: str, opensmile_best_by_target: pd.DataFrame) -> List[str]:
    names = []
    if cfg.model_mode == "best_opensmile_per_target" and not opensmile_best_by_target.empty:
        match = opensmile_best_by_target[opensmile_best_by_target["target"].eq(target)]
        if not match.empty:
            names.append(str(match.iloc[0]["model"]))
    elif cfg.model_mode == "compact_panel":
        names.extend(["Ridge", "ExtraTrees", "RandomForest", "GradientBoosting"])
    if cfg.include_ridge_reference:
        names.append("Ridge")
    if cfg.include_dummy_reference:
        names.append("DummyMean")
    return list(dict.fromkeys(names or ["Ridge"]))


def clean_xy(cfg: PipelineConfig, df_in: pd.DataFrame, features: List[str], target: str) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    existing = [f for f in dict.fromkeys(features) if f in df_in.columns]
    work_cols = existing + [target, cfg.song_id_col]
    work = df_in[work_cols].copy()
    work[target] = pd.to_numeric(work[target], errors="coerce")
    work = work.dropna(subset=[target, cfg.song_id_col]).reset_index(drop=True)
    X = work[existing].apply(pd.to_numeric, errors="coerce")
    X = X.drop(columns=X.columns[X.isna().all()], errors="ignore")
    X = X.drop(columns=X.columns[X.nunique(dropna=True) <= 1], errors="ignore")
    y = work[target]
    return X, y, work


def audit_feature_leakage(features: List[str]) -> None:
    leak_patterns = [r"song_id", r"block_idx", r"start_time", r"end_time", r"duration", r"valence", r"arousal", r"quadrant", r"title", r"artist", r"genre", r"target"]
    suspicious = []
    for feature in features:
        feature_lower = str(feature).lower()
        if any(re.search(pattern, feature_lower) for pattern in leak_patterns):
            suspicious.append(feature)
    if suspicious:
        raise ValueError(f"Features suspeitas de vazamento: {suspicious[:30]}")


def make_model_pipeline(model, needs_scaling: bool, selector_k: Optional[int], n_features: int) -> Pipeline:
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if selector_k is not None and n_features > 1:
        steps.append(("selector", SelectKBest(score_func=f_regression, k=min(selector_k, n_features))))
    if needs_scaling:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", clone(model)))
    return Pipeline(steps)


def evaluate_feature_set_with_folds(cfg: PipelineConfig, df_in: pd.DataFrame, feature_set_name: str, features: List[str], target: str, model_name: str, selector_k: Optional[int] = None) -> Tuple[Dict[str, object], pd.DataFrame]:
    X, y, work = clean_xy(cfg, df_in, features, target)
    audit_feature_leakage(features)
    selector_name = "none" if selector_k is None else f"SelectKBest(k={selector_k})"
    requested_features = int(len(dict.fromkeys(features)))

    if X.empty:
        summary = {
            "feature_set": feature_set_name,
            "target": target,
            "model": model_name,
            "selector": selector_name,
            "cv": "no_valid_features",
            "n_samples": int(y.shape[0]),
            "n_songs": int(work[cfg.song_id_col].nunique()) if cfg.song_id_col in work.columns else 0,
            "n_features_requested": requested_features,
            "n_features_effective": 0,
            "n_features": 0,
            "rmse_mean": np.nan,
            "rmse_std": np.nan,
            "mae_mean": np.nan,
            "mae_std": np.nan,
            "r2_mean": np.nan,
            "r2_std": np.nan,
            "pearson_mean": np.nan,
            "pearson_std": np.nan,
            "status": "no_valid_features",
        }
        return summary, pd.DataFrame()

    n_groups = work[cfg.song_id_col].nunique()
    if n_groups >= cfg.n_splits:
        cv = GroupKFold(n_splits=cfg.n_splits)
        split_kw = {"groups": work[cfg.song_id_col].values}
        cv_name = f"GroupKFold({cfg.n_splits}) by song"
    else:
        cv = KFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.random_state)
        split_kw = {}
        cv_name = f"KFold({cfg.n_splits}) fallback"

    model, needs_scaling = model_factory(cfg, model_name)
    pipe = make_model_pipeline(model, needs_scaling=needs_scaling, selector_k=selector_k, n_features=X.shape[1])

    fold_rows = []
    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y, **split_kw), start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        fitted = clone(pipe)
        fitted.fit(X_train, y_train)
        y_pred = fitted.predict(X_test)

        rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
        mae = float(mean_absolute_error(y_test, y_pred))
        r2 = float(r2_score(y_test, y_pred))
        y_test_std = float(np.std(y_test))
        y_pred_std = float(np.std(y_pred))
        pearson = float(np.corrcoef(y_test, y_pred)[0, 1]) if y_test_std > 0 and y_pred_std > 0 else np.nan

        fold_rows.append({
            "feature_set": feature_set_name,
            "target": target,
            "model": model_name,
            "selector": selector_name if selector_k is None else f"SelectKBest(k={min(selector_k, X.shape[1])})",
            "cv": cv_name,
            "fold": fold_idx,
            "n_train": int(len(train_idx)),
            "n_test": int(len(test_idx)),
            "n_samples": int(X.shape[0]),
            "n_songs": int(n_groups),
            "n_features_requested": requested_features,
            "n_features_effective": int(X.shape[1]),
            "n_features": int(X.shape[1]),
            "rmse": rmse,
            "mae": mae,
            "r2": r2,
            "pearson": pearson,
            "status": "ok",
        })

    fold_df = pd.DataFrame(fold_rows)
    effective_k = int(X.shape[1] if selector_k is None else min(selector_k, X.shape[1]))
    summary = {
        "feature_set": feature_set_name,
        "target": target,
        "model": model_name,
        "selector": selector_name if selector_k is None else f"SelectKBest(k={effective_k})",
        "cv": cv_name,
        "n_samples": int(X.shape[0]),
        "n_songs": int(n_groups),
        "n_features_requested": requested_features,
        "n_features_effective": int(X.shape[1]),
        "n_features": int(X.shape[1]),
        "rmse_mean": float(fold_df["rmse"].mean()) if not fold_df.empty else np.nan,
        "rmse_std": float(fold_df["rmse"].std(ddof=0)) if not fold_df.empty else np.nan,
        "mae_mean": float(fold_df["mae"].mean()) if not fold_df.empty else np.nan,
        "mae_std": float(fold_df["mae"].std(ddof=0)) if not fold_df.empty else np.nan,
        "r2_mean": float(fold_df["r2"].mean()) if not fold_df.empty else np.nan,
        "r2_std": float(fold_df["r2"].std(ddof=0)) if not fold_df.empty else np.nan,
        "pearson_mean": float(fold_df["pearson"].mean()) if not fold_df.empty else np.nan,
        "pearson_std": float(fold_df["pearson"].std(ddof=0)) if not fold_df.empty else np.nan,
        "status": "ok",
    }
    return summary, fold_df


def evaluate_feature_set(cfg: PipelineConfig, df_in: pd.DataFrame, feature_set_name: str, features: List[str], target: str, model_name: str, selector_k: Optional[int] = None) -> Dict[str, object]:
    summary, _ = evaluate_feature_set_with_folds(cfg, df_in, feature_set_name, features, target, model_name, selector_k=selector_k)
    return summary


def build_fold_level_comparison(fold_results: pd.DataFrame, baseline_feature_set: str, challenger_feature_set: str, target: str) -> pd.DataFrame:
    if fold_results.empty:
        return pd.DataFrame()
    base = fold_results[(fold_results["feature_set"].eq(baseline_feature_set)) & (fold_results["target"].eq(target)) & (fold_results["status"].eq("ok"))].copy()
    chal = fold_results[(fold_results["feature_set"].eq(challenger_feature_set)) & (fold_results["target"].eq(target)) & (fold_results["status"].eq("ok"))].copy()
    if base.empty or chal.empty:
        return pd.DataFrame()

    join_cols = [c for c in ["fold", "model", "selector"] if c in base.columns and c in chal.columns]
    if "fold" not in join_cols:
        return pd.DataFrame()
    paired = base.merge(chal, on=join_cols, suffixes=("_baseline", "_challenger"))
    if paired.empty:
        return pd.DataFrame()

    paired["delta_rmse"] = paired["rmse_challenger"] - paired["rmse_baseline"]
    paired["delta_mae"] = paired["mae_challenger"] - paired["mae_baseline"]
    paired["delta_r2"] = paired["r2_challenger"] - paired["r2_baseline"]
    paired["delta_pearson"] = paired["pearson_challenger"] - paired["pearson_baseline"]

    summary = pd.DataFrame([{
        "target": target,
        "baseline_feature_set": baseline_feature_set,
        "challenger_feature_set": challenger_feature_set,
        "n_folds": int(paired.shape[0]),
        "delta_rmse_mean": float(paired["delta_rmse"].mean()),
        "delta_rmse_std": float(paired["delta_rmse"].std(ddof=0)),
        "delta_mae_mean": float(paired["delta_mae"].mean()),
        "delta_r2_mean": float(paired["delta_r2"].mean()),
        "delta_pearson_mean": float(paired["delta_pearson"].mean()),
        "paired_ttest_rmse": None if ttest_rel is None else float(ttest_rel(paired["rmse_challenger"], paired["rmse_baseline"], nan_policy="omit").pvalue),
        "paired_wilcoxon_rmse": None if wilcoxon is None else float(wilcoxon(paired["rmse_challenger"], paired["rmse_baseline"]).pvalue),
    }])
    return summary


def run_experiments(cfg: PipelineConfig, modeling_df: pd.DataFrame, feature_scenarios: Dict[str, List[str]], opensmile_best_by_target: pd.DataFrame) -> pd.DataFrame:
    all_rows = []
    all_fold_rows = []
    if not cfg.run_fingerprint_experiments:
        return pd.DataFrame()
    for feature_set_name, features in progress_iter(feature_scenarios.items(), desc="Feature scenarios", total=len(feature_scenarios)):
        for target in cfg.targets:
            selector_options: List[Optional[int]] = [None]
            if cfg.run_selectkbest_inside_cv and len(features) > cfg.selector_k:
                selector_options.append(cfg.selector_k)
            for model_name in model_names_for_target(cfg, target, opensmile_best_by_target):
                for selector_k in selector_options:
                    summary, fold_df = evaluate_feature_set_with_folds(cfg, modeling_df, feature_set_name, features, target, model_name, selector_k=selector_k)
                    all_rows.append(summary)
                    if not fold_df.empty:
                        all_fold_rows.append(fold_df)
    results = pd.DataFrame(all_rows)
    fold_results = pd.concat(all_fold_rows, ignore_index=True) if all_fold_rows else pd.DataFrame()
    if not results.empty:
        results = results.sort_values(["target", "rmse_mean", "feature_set", "model", "selector"], na_position="last").reset_index(drop=True)
    if not fold_results.empty:
        fold_results = fold_results.sort_values(["target", "feature_set", "model", "selector", "fold"]).reset_index(drop=True)
    save_table(cfg, results, "fingerprint_vs_opensmile_feature_set_results.csv")
    save_table(cfg, fold_results, "fingerprint_vs_opensmile_feature_set_fold_results.csv")
    return results


def build_tcc_summary(cfg: PipelineConfig, opensmile_best_by_target: pd.DataFrame, best_new_by_target: pd.DataFrame, comparison_vs_opensmile: pd.DataFrame, opensmile_block_cache: pd.DataFrame) -> str:
    fingerprint_only = not cfg.use_cached_opensmile_results and not cfg.rebuild_opensmile_block_cache
    if fingerprint_only:
        lines = ["Pipeline em modo fingerprint-only: openSMILE foi pulado nesta rodada."]
    else:
        lines = ["Pipeline refatorado em modo cache-first: openSMILE permanece opcional e so entra quando habilitado na configuracao."]

    lines.append("Os cenarios avaliados usam apenas os conjuntos de features de fingerprint disponiveis no inventario.")

    if cfg.use_cached_opensmile_results and not opensmile_best_by_target.empty:
        for _, row in opensmile_best_by_target.iterrows():
            lines.append(
                f"Baseline openSMILE salvo para {row['target']}: {row['model']} "
                f"com RMSE={row['rmse_mean']:.4f}, R2={row['r2_mean']:.4f} e {int(row['n_features'])} features."
            )
    elif cfg.use_cached_opensmile_results:
        lines.append("Baseline openSMILE salvo nao foi encontrado para comparacao direta.")
    else:
        lines.append("Baseline e diagnosticos openSMILE nao foram carregados porque use_cached_opensmile_results=False.")

    comparison_cols = {"target", "feature_set", "model", "selector", "delta_rmse_vs_openSMILE", "impacto_vs_openSMILE"}
    has_opensmile_comparison = not comparison_vs_opensmile.empty and comparison_cols.issubset(comparison_vs_opensmile.columns)

    if best_new_by_target.empty:
        lines.append("Nenhum resultado novo de fingerprint foi gerado nesta execucao.")
    else:
        for _, row in best_new_by_target.iterrows():
            metric_parts = [f"RMSE={row['rmse_mean']:.4f}"]
            if "r2_mean" in row.index and pd.notna(row.get("r2_mean")):
                metric_parts.append(f"R2={row['r2_mean']:.4f}")
            if "gain_vs_dummy_pct" in row.index and pd.notna(row.get("gain_vs_dummy_pct")):
                metric_parts.append(f"ganho_vs_dummy={row['gain_vs_dummy_pct']:.1f}%")

            base_line = (
                f"Melhor cenario de fingerprint para {row['target']}: {row['feature_set']} "
                f"com {row['model']} ({row['selector']}), " + ", ".join(metric_parts)
            )

            if has_opensmile_comparison:
                comp = comparison_vs_opensmile[
                    (comparison_vs_opensmile["target"].eq(row["target"]))
                    & (comparison_vs_opensmile["feature_set"].eq(row["feature_set"]))
                    & (comparison_vs_opensmile["model"].eq(row["model"]))
                    & (comparison_vs_opensmile["selector"].eq(row["selector"]))
                ]
                if not comp.empty and pd.notna(comp.iloc[0].get("delta_rmse_vs_openSMILE")):
                    delta = comp.iloc[0]["delta_rmse_vs_openSMILE"]
                    impact = comp.iloc[0]["impacto_vs_openSMILE"]
                    base_line += f"; isso {impact} em {abs(delta):.4f} RMSE contra o baseline openSMILE salvo"

            lines.append(base_line + ".")

    if fingerprint_only:
        lines.append("Cache openSMILE por bloco nao foi carregado nem reconstruido nesta rodada.")
    elif opensmile_block_cache.empty:
        lines.append("O cache openSMILE por bloco nao foi usado nesta comparacao.")
    else:
        lines.append("O cache openSMILE por bloco foi carregado, mas os cenarios de fingerprint continuam separados no inventario.")

    summary_text = "\n".join(lines)
    summary_path = cfg.tables_dir / "fingerprint_vs_opensmile_summary_for_tcc.txt"
    summary_path.write_text(summary_text, encoding="utf-8")
    print(summary_text)
    print(f"\nSaved summary: {summary_path}")
    return summary_text


In [11]:
def run_pipeline(cfg: Optional[PipelineConfig] = None) -> Dict[str, object]:
    cfg = cfg or PipelineConfig()
    cfg.ensure_dirs()

    if cfg.run_pycaret:
        print("[INFO] run_pycaret=True is intentionally ignored here; this refactor is scikit/cache-first.")

    if cfg.use_cached_opensmile_results:
        opensmile_reference_results, opensmile_best_by_target, opensmile_selected_features = load_opensmile_reference(cfg)
    else:
        opensmile_reference_results = pd.DataFrame()
        opensmile_best_by_target = pd.DataFrame()
        opensmile_selected_features = pd.DataFrame()
        print("[INFO] openSMILE baseline pulado; executando apenas fingerprints.")

    fingerprint_df, fingerprint_feature_sets, fingerprint_inventory = load_fingerprint_dataset(cfg)
    print(f"Fingerprint dataset: {fingerprint_df.shape[0]} blocks x {fingerprint_df.shape[1]} columns")

    if cfg.use_cached_opensmile_results or cfg.rebuild_opensmile_block_cache:
        opensmile_block_cache, opensmile_block_features = load_or_build_opensmile_block_cache(
            cfg, fingerprint_df, opensmile_selected_features
        )
        print(f"openSMILE block-cache features: {len(opensmile_block_features)}")
    else:
        opensmile_block_cache = pd.DataFrame()
        opensmile_block_features = []

    modeling_df, feature_scenarios, scenario_inventory = assemble_feature_scenarios(
        cfg,
        fingerprint_df,
        fingerprint_feature_sets,
        opensmile_block_cache,
        opensmile_block_features,
    )

    fingerprint_results = run_experiments(cfg, modeling_df, feature_scenarios, opensmile_best_by_target)
    opensmile_block_reference = pd.DataFrame()
    if (
        (cfg.use_cached_opensmile_results or cfg.rebuild_opensmile_block_cache)
        and not fingerprint_results.empty
        and "openSMILE Selecionadas (bloco cache)" in fingerprint_results["feature_set"].values
    ):
        block_rows = fingerprint_results[
            fingerprint_results["feature_set"].eq("openSMILE Selecionadas (bloco cache)")
            & fingerprint_results["status"].eq("ok")
        ]
        if not block_rows.empty:
            opensmile_block_reference = block_rows.sort_values(["target", "rmse_mean"]).groupby("target", as_index=False).first()

    comparison_reference = opensmile_block_reference if not opensmile_block_reference.empty else opensmile_best_by_target
    if not comparison_reference.empty:
        comparison_vs_opensmile = compare_against_opensmile_reference(fingerprint_results, comparison_reference)
        save_table(cfg, comparison_vs_opensmile, "delta_vs_cached_opensmile_baseline.csv")
    else:
        comparison_vs_opensmile = pd.DataFrame()

    best_by_feature_set, best_new_by_target = best_result_tables(cfg, fingerprint_results)
    figures = make_plots(cfg, best_by_feature_set, comparison_vs_opensmile)
    summary_text = build_tcc_summary(
        cfg,
        opensmile_best_by_target,
        best_new_by_target,
        comparison_vs_opensmile,
        opensmile_block_cache,
    )

    return {
        "config": cfg,
        "opensmile_reference_results": opensmile_reference_results,
        "opensmile_best_by_target": opensmile_best_by_target,
        "opensmile_block_reference": opensmile_block_reference,
        "opensmile_selected_features": opensmile_selected_features,
        "fingerprint_df": fingerprint_df,
        "fingerprint_inventory": fingerprint_inventory,
        "opensmile_block_cache": opensmile_block_cache,
        "opensmile_block_features": opensmile_block_features,
        "modeling_df": modeling_df,
        "feature_scenarios": feature_scenarios,
        "scenario_inventory": scenario_inventory,
        "fingerprint_results": fingerprint_results,
        "comparison_vs_opensmile": comparison_vs_opensmile,
        "best_by_feature_set": best_by_feature_set,
        "best_new_by_target": best_new_by_target,
        "figures": figures,
        "summary_text": summary_text,
    }


In [12]:
artifacts = run_pipeline(cfg)


[INFO] openSMILE baseline pulado; executando apenas fingerprints.
Saved table: C:\dev\python\TCC Ajustado\Dados\pycaret_fingerprint_outputs\fingerprint_vs_opensmile_refactor\tables\fingerprint_feature_inventory.csv
Fingerprint dataset: 6461 blocks x 455 columns
Saved table: C:\dev\python\TCC Ajustado\Dados\pycaret_fingerprint_outputs\fingerprint_vs_opensmile_refactor\tables\feature_scenario_inventory.csv


Feature scenarios:   0%|          | 0/4 [00:00<?, ?it/s]

Saved table: C:\dev\python\TCC Ajustado\Dados\pycaret_fingerprint_outputs\fingerprint_vs_opensmile_refactor\tables\fingerprint_vs_opensmile_feature_set_results.csv
Saved table: C:\dev\python\TCC Ajustado\Dados\pycaret_fingerprint_outputs\fingerprint_vs_opensmile_refactor\tables\fingerprint_vs_opensmile_feature_set_fold_results.csv
Saved table: C:\dev\python\TCC Ajustado\Dados\pycaret_fingerprint_outputs\fingerprint_vs_opensmile_refactor\tables\best_result_by_target_and_feature_set.csv
Saved table: C:\dev\python\TCC Ajustado\Dados\pycaret_fingerprint_outputs\fingerprint_vs_opensmile_refactor\tables\best_fingerprint_scenario_by_target.csv


Pipeline em modo fingerprint-only: openSMILE foi pulado nesta rodada.
Os cenarios avaliados usam apenas os conjuntos de features de fingerprint disponiveis no inventario.
Baseline e diagnosticos openSMILE nao foram carregados porque use_cached_opensmile_results=False.
Melhor cenario de fingerprint para arousal: Band Rank (bloco) com GradientBoosting (none), RMSE=0.1888, R2=0.5437, ganho_vs_dummy=32.7%.
Melhor cenario de fingerprint para valence: Rank (bloco) com ExtraTrees (none), RMSE=0.2223, R2=0.1930, ganho_vs_dummy=11.0%.
Cache openSMILE por bloco nao foi carregado nem reconstruido nesta rodada.

Saved summary: C:\dev\python\TCC Ajustado\Dados\pycaret_fingerprint_outputs\fingerprint_vs_opensmile_refactor\tables\fingerprint_vs_opensmile_summary_for_tcc.txt


## 10. Correla??o dos fingerprints com Valence e Arousal

Mesmo estilo da se??o de correla??o do notebook openSMILE original: calcula Pearson/Spearman entre cada feature de fingerprint e cada target, exibe as Top-25 tabelas, plota Top-20 correla??es de Pearson e gera heatmaps com as features mais correlacionadas.


In [13]:
from IPython.display import Markdown, display


def short_fingerprint_feature_name(feature: str, max_len: int = 72) -> str:
    text = (
        str(feature)
        .replace("fpband__", "band::")
        .replace("fprank__", "rank::")
        .replace("rawpeaks_band__", "raw_band::")
        .replace("rawpeaks_rank__", "raw_rank::")
    )
    if len(text) <= max_len:
        return text
    return text[: max_len - 3] + "..."



def fingerprint_feature_family_key(feature: str) -> str:
    feature = str(feature)
    for prefix in ("fpband__", "fprank__", "rawpeaks_band__", "rawpeaks_rank__"):
        if feature.startswith(prefix):
            return prefix
    return "other"



_FAMILY_LABELS = {
    "fpband__": "Band Rank (bloco)",
    "fprank__": "Rank (bloco)",
    "rawpeaks_band__": "Band Rank (raw peaks)",
    "rawpeaks_rank__": "Rank (raw peaks)",
}

def fingerprint_feature_family(feature: str) -> str:
    return _FAMILY_LABELS.get(fingerprint_feature_family_key(feature), "Outras")



def fingerprint_features_by_family(feature_cols: List[str]) -> Dict[str, List[str]]:
    grouped = {k: [] for k in _FAMILY_LABELS}
    for col in dict.fromkeys(feature_cols):
        family_key = fingerprint_feature_family_key(col)
        if family_key in grouped:
            grouped[family_key].append(col)
    return grouped



def fingerprint_frequency_group(feature: str) -> str:
    raw = str(feature).replace("fpband__", "").replace("fprank__", "").replace("rawpeaks_band__", "").replace("rawpeaks_rank__", "").lower()
    if "very_high" in raw:
        return "very_high"
    if re.search(r"(^|_)high(_|$)", raw):
        return "high"
    if re.search(r"(^|_)mid(_|$)", raw):
        return "mid"
    if re.search(r"(^|_)low(_|$)", raw):
        return "low"
    if "top_" in raw or "top" in raw:
        return "top_rank"
    if raw.startswith("n_") or "event_count" in raw or "peak" in raw or "octave_duplicate" in raw:
        return "counts"
    return "global"



def fingerprint_metric_group(feature: str) -> str:
    raw = str(feature).replace("fpband__", "").replace("fprank__", "").replace("rawpeaks_band__", "").replace("rawpeaks_rank__", "").lower()
    if "freq" in raw:
        return "frequency"
    if "mag" in raw or "magnitude" in raw:
        return "magnitude"
    if "rank" in raw:
        return "rank"
    if "midi" in raw or "semitone" in raw or "pitch" in raw or "octave" in raw:
        return "pitch_octave"
    if "count" in raw or raw.startswith("n_") or "event" in raw or "peak" in raw:
        return "counts"
    if "std" in raw:
        return "dispersion"
    if "mean" in raw:
        return "central_tendency"
    return "other"



def fingerprint_target_correlation_report(
    df_in: pd.DataFrame,
    feature_cols: List[str],
    target_col: str,
    family_key: Optional[str] = None,
    family_label: Optional[str] = None,
) -> pd.DataFrame:
    rows = []
    feature_cols = list(dict.fromkeys([c for c in feature_cols if c in df_in.columns]))

    for col in feature_cols:
        x = pd.to_numeric(df_in[col], errors="coerce")
        y = pd.to_numeric(df_in[target_col], errors="coerce")
        valid = x.notna() & y.notna()

        if valid.sum() < 5:
            continue

        pearson = x[valid].corr(y[valid], method="pearson")
        spearman = x[valid].corr(y[valid], method="spearman")

        rows.append({
            "feature": col,
            "feature_label": short_fingerprint_feature_name(col),
            "fingerprint_family": family_label or fingerprint_feature_family(col),
            "fingerprint_family_key": family_key or fingerprint_feature_family_key(col),
            "categoria_fingerprint": fingerprint_frequency_group(col),
            "metric_group": fingerprint_metric_group(col),
            "target": target_col,
            "pearson": pearson,
            "abs_pearson": abs(pearson) if pd.notna(pearson) else np.nan,
            "spearman": spearman,
            "abs_spearman": abs(spearman) if pd.notna(spearman) else np.nan,
            "n_valid": int(valid.sum()),
        })

    if not rows:
        return pd.DataFrame(columns=[
            "feature",
            "feature_label",
            "fingerprint_family",
            "fingerprint_family_key",
            "categoria_fingerprint",
            "metric_group",
            "target",
            "pearson",
            "abs_pearson",
            "spearman",
            "abs_spearman",
            "n_valid",
        ])

    return pd.DataFrame(rows).sort_values("abs_pearson", ascending=False).reset_index(drop=True)



def plot_fingerprint_target_correlation_bars(
    cfg: PipelineConfig,
    corr_df: pd.DataFrame,
    target_col: str,
    family_label: str,
    top_n: int = 20,
):
    if corr_df.empty:
        print(f"Nenhuma correlacao calculada para {target_col} ({family_label}); grafico ignorado.")
        return None

    top_corr = corr_df.head(top_n).copy().sort_values("pearson")
    height = max(540, 28 * len(top_corr) + 180)
    fig = px.bar(
        top_corr,
        x="pearson",
        y="feature_label",
        orientation="h",
        color="metric_group",
        hover_data=["feature", "fingerprint_family", "categoria_fingerprint", "spearman", "n_valid"],
        title=f"Top-{top_n} correlaÃ§Ãµes de Pearson - {family_label} vs {target_col}",
    )
    fig.add_vline(x=0, line_dash="dash", line_color="black")
    fig.update_layout(
        template="plotly_white",
        xaxis_title="Pearson",
        yaxis_title="",
        height=height,
        margin=dict(l=20, r=20, t=80, b=20),
    )
    fig.update_yaxes(automargin=True)
    if cfg.show_figures:
        fig.show()
    save_plot(cfg, fig, f"fingerprint_top{top_n}_pearson_{family_label.replace(' ', '_').lower()}_{target_col}")
    return fig



def plot_fingerprint_correlation_heatmap(
    cfg: PipelineConfig,
    df_in: pd.DataFrame,
    corr_df: pd.DataFrame,
    target_col: str,
    family_label: str,
    top_n: int = 10,
):
    if corr_df.empty:
        print(f"Nenhuma correlacao calculada para {target_col} ({family_label}); heatmap ignorado.")
        return None

    feats = [f for f in corr_df.head(top_n)["feature"].tolist() if f in df_in.columns]
    if not feats:
        print(f"Nenhuma feature top-{top_n} encontrada no DataFrame para {target_col} ({family_label}); heatmap ignorado.")
        return None

    work = df_in[feats + [target_col]].apply(pd.to_numeric, errors="coerce")
    corr = work.corr()
    label_map = {f: short_fingerprint_feature_name(f, max_len=46) for f in feats}
    corr = corr.rename(index=label_map, columns=label_map)

    fig = px.imshow(
        corr,
        aspect="auto",
        title=f"CorrelaÃ§Ãµes entre top-{len(feats)} features de {family_label} e {target_col}",
        color_continuous_scale="RdBu_r",
        zmin=-1,
        zmax=1,
    )
    fig.update_layout(template="plotly_white", height=720, margin=dict(l=20, r=20, t=80, b=20))
    if cfg.show_figures:
        fig.show()
    save_plot(cfg, fig, f"fingerprint_correlation_heatmap_{family_label.replace(' ', '_').lower()}_{target_col}")
    return fig



def build_fingerprint_correlation_summary(corr_all: pd.DataFrame) -> pd.DataFrame:
    if corr_all.empty:
        return pd.DataFrame()
    summary = (
        corr_all
        .groupby(["target", "fingerprint_family", "fingerprint_family_key", "categoria_fingerprint", "metric_group"], dropna=False)
        .agg(
            n_features=("feature", "nunique"),
            mean_abs_pearson=("abs_pearson", "mean"),
            max_abs_pearson=("abs_pearson", "max"),
            mean_abs_spearman=("abs_spearman", "mean"),
            max_abs_spearman=("abs_spearman", "max"),
        )
        .reset_index()
        .sort_values(["target", "fingerprint_family", "max_abs_pearson"], ascending=[True, True, False])
    )
    return summary



def run_fingerprint_correlation_analysis(
    cfg: PipelineConfig,
    fingerprint_df: pd.DataFrame,
    feature_scenarios: Dict[str, List[str]],
    top_n: int = 20,
    heatmap_top_n: int = 10,
) -> Dict[str, object]:
    fingerprint_features = list(dict.fromkeys([feature for features in feature_scenarios.values() for feature in features]))
    if not fingerprint_features:
        fingerprint_features = [
            c for c in fingerprint_df.columns
            if any(c.startswith(p) for p in _FAMILY_LABELS)
        ]

    family_map = fingerprint_features_by_family(fingerprint_features)
    family_labels = _FAMILY_LABELS

    reports = {}
    figures = {}
    heatmaps = {}
    corr_frames = []

    for target_col in cfg.targets:
        display(Markdown(f"### CorrelaÃ§Ãµes dos fingerprints com {target_col}"))
        target_reports = {}

        for family_key, family_features in family_map.items():
            family_label = family_labels[family_key]
            if not family_features:
                continue

            corr_df = fingerprint_target_correlation_report(
                fingerprint_df,
                family_features,
                target_col,
                family_key=family_key,
                family_label=family_label,
            )
            if corr_df.empty:
                continue

            target_reports[family_key] = corr_df
            corr_frames.append(corr_df)

            save_table(cfg, corr_df, f"corr_fingerprint_features_{family_key}_{target_col}.csv")

            display(Markdown(f"#### {family_label}"))
            display(corr_df.head(25))

            figures[(target_col, family_key)] = plot_fingerprint_target_correlation_bars(
                cfg,
                corr_df,
                target_col,
                family_label,
                top_n=top_n,
            )
            heatmaps[(target_col, family_key)] = plot_fingerprint_correlation_heatmap(
                cfg,
                fingerprint_df,
                corr_df,
                target_col,
                family_label,
                top_n=heatmap_top_n,
            )

        reports[target_col] = target_reports

    corr_all = pd.concat(corr_frames, ignore_index=True) if corr_frames else pd.DataFrame()
    summary = build_fingerprint_correlation_summary(corr_all)

    save_table(cfg, corr_all, "corr_fingerprint_features_all_targets.csv")
    save_table(cfg, summary, "corr_fingerprint_summary_by_category.csv")

    display(Markdown("### Resumo das correlaÃ§Ãµes dos fingerprints por tÃ©cnica"))
    display(summary)

    return {
        "reports": reports,
        "all_correlations": corr_all,
        "summary_by_category": summary,
        "bar_figures": figures,
        "heatmaps": heatmaps,
    }


fingerprint_correlation_artifacts = run_fingerprint_correlation_analysis(
    cfg,
    artifacts["fingerprint_df"],
    artifacts["feature_scenarios"],
    top_n=20,
    heatmap_top_n=10,
)

artifacts["fingerprint_correlations"] = fingerprint_correlation_artifacts


### CorrelaÃ§Ãµes dos fingerprints com valence

Saved table: C:\dev\python\TCC Ajustado\Dados\pycaret_fingerprint_outputs\fingerprint_vs_opensmile_refactor\tables\corr_fingerprint_features_fpband___valence.csv


#### Band Rank (bloco)

,feature,feature_label,fingerprint_family,fingerprint_family_key,categoria_fingerprint,metric_group,target,pearson,abs_pearson,spearman,abs_spearman,n_valid
0,fpband__energy_db_very_high,band::energy_db_very_high,Band Rank (bloco),fpband__,very_high,other,valence,0.291232,0.291232,0.336196,0.336196,6461
1,fpband__energy_db_high,band::energy_db_high,Band Rank (bloco),fpband__,high,other,valence,0.264516,0.264516,0.293746,0.293746,6461
2,fpband__energy_norm_very_high,band::energy_norm_very_high,Band Rank (bloco),fpband__,very_high,other,valence,0.200697,0.200697,0.327019,0.327019,6461
3,fpband__score_very_high,band::score_very_high,Band Rank (bloco),fpband__,very_high,other,valence,0.189899,0.189899,0.327610,0.327610,6461
4,fpband__fp_mid_mag_std_norm_block,band::fp_mid_mag_std_norm_block,Band Rank (bloco),fpband__,mid,magnitude,valence,-0.155429,0.155429,-0.167409,0.167409,6461
5,fpband__energy_db_mid,band::energy_db_mid,Band Rank (bloco),fpband__,mid,other,valence,0.148389,0.148389,0.152521,0.152521,6461
6,fpband__energy_db_low,band::energy_db_low,Band Rank (bloco),fpband__,low,other,valence,0.144664,0.144664,0.140686,0.140686,6461
7,fpband__fp_very_high_top_10_magnitude_norm_block,band::fp_very_high_top_10_magnitude_norm_block,Band Rank (bloco),fpband__,very_high,magnitude,valence,0.137854,0.137854,0.309314,0.309314,6461
8,fpband__fp_very_high_top_9_magnitude_norm_block,band::fp_very_high_top_9_magnitude_norm_block,Band Rank (bloco),fpband__,very_high,magnitude,valence,0.136559,0.136559,0.309265,0.309265,6461
9,fpband__fp_very_high_top_8_magnitude_norm_block,band::fp_very_high_top_8_magnitude_norm_block,Band Rank (bloco),fpband__,very_high,magnitude,valence,0.134979,0.134979,0.309195,0.309195,6461


Saved table: C:\dev\python\TCC Ajustado\Dados\pycaret_fingerprint_outputs\fingerprint_vs_opensmile_refactor\tables\corr_fingerprint_features_fprank___valence.csv


...

#### Rank (raw peaks)

,feature,feature_label,fingerprint_family,fingerprint_family_key,categoria_fingerprint,metric_group,target,pearson,abs_pearson,spearman,abs_spearman,n_valid
0,rawpeaks_rank__midi_note,raw_rank::midi_note,Rank (raw peaks),rawpeaks_rank__,global,pitch_octave,arousal,-0.238878,0.238878,-0.240525,0.240525,6461
1,rawpeaks_rank__semitone,raw_rank::semitone,Rank (raw peaks),rawpeaks_rank__,global,pitch_octave,arousal,-0.238664,0.238664,-0.240643,0.240643,6461
2,rawpeaks_rank__octave,raw_rank::octave,Rank (raw peaks),rawpeaks_rank__,global,pitch_octave,arousal,-0.235562,0.235562,-0.237063,0.237063,6461
3,rawpeaks_rank__frequency_hz,raw_rank::frequency_hz,Rank (raw peaks),rawpeaks_rank__,global,frequency,arousal,-0.124497,0.124497,-0.216685,0.216685,6461
4,rawpeaks_rank__pitch_class,raw_rank::pitch_class,Rank (raw peaks),rawpeaks_rank__,global,pitch_octave,arousal,-0.112415,0.112415,-0.110021,0.110021,6461
5,rawpeaks_rank__peak_time_sec,raw_rank::peak_time_sec,Rank (raw peaks),rawpeaks_rank__,counts,counts,arousal,-0.039535,0.039535,-0.033538,0.033538,6461
6,rawpeaks_rank__peak_rank_global,raw_rank::peak_rank_global,Rank (raw peaks),rawpeaks_rank__,counts,rank,arousal,NaN,NaN,NaN,NaN,6461


Saved table: C:\dev\python\TCC Ajustado\Dados\pycaret_fingerprint_outputs\fingerprint_vs_opensmile_refactor\tables\corr_fingerprint_features_all_targets.csv
Saved table: C:\dev\python\TCC Ajustado\Dados\pycaret_fingerprint_outputs\fingerprint_vs_opensmile_refactor\tables\corr_fingerprint_summary_by_category.csv


### Resumo das correlaÃ§Ãµes dos fingerprints por tÃ©cnica

,target,fingerprint_family,fingerprint_family_key,categoria_fingerprint,metric_group,n_features,mean_abs_pearson,max_abs_pearson,mean_abs_spearman,max_abs_spearman
4,arousal,Band Rank (bloco),fpband__,high,other,3,0.565967,0.695876,0.639109,0.703429
16,arousal,Band Rank (bloco),fpband__,very_high,other,3,0.488608,0.608262,0.575844,0.628013
12,arousal,Band Rank (bloco),fpband__,mid,other,3,0.335411,0.539504,0.315463,0.545219
3,arousal,Band Rank (bloco),fpband__,high,magnitude,12,0.394488,0.465696,0.503161,0.567825
7,arousal,Band Rank (bloco),fpband__,low,magnitude,12,0.355085,0.439006,0.368602,0.459373
...,...,...,...,...,...,...,...,...,...,...
61,valence,Rank (bloco),fprank__,counts,pitch_octave,2,0.005086,0.005086,0.027861,0.027861
68,valence,Rank (raw peaks),rawpeaks_rank__,counts,counts,1,0.166312,0.166312,0.099781,0.099781
71,valence,Rank (raw peaks),rawpeaks_rank__,global,pitch_octave,4,0.144132,0.155212,0.158737,0.173371
70,valence,Rank (raw peaks),rawpeaks_rank__,global,frequency,1,0.075480,0.075480,0.160772,0.160772


In [14]:
def compare_against_opensmile_reference(results: pd.DataFrame, reference: pd.DataFrame) -> pd.DataFrame:
    if results.empty or reference.empty:
        return pd.DataFrame()

    ref_cols = ["target", "model", "rmse_mean", "mae_mean", "r2_mean", "n_features", "cv"]
    ref = reference[[c for c in ref_cols if c in reference.columns]].copy()
    ref = ref.rename(columns={
        "model": "opensmile_reference_model",
        "rmse_mean": "opensmile_reference_rmse",
        "mae_mean": "opensmile_reference_mae",
        "r2_mean": "opensmile_reference_r2",
        "n_features": "opensmile_reference_n_features",
        "cv": "opensmile_reference_cv",
    })

    comp = results[results["status"].eq("ok")].merge(ref, on="target", how="left")
    comp["delta_rmse_vs_openSMILE"] = comp["rmse_mean"] - comp["opensmile_reference_rmse"]
    comp["delta_rmse_pct_vs_openSMILE"] = 100.0 * comp["delta_rmse_vs_openSMILE"] / comp["opensmile_reference_rmse"]
    comp["impacto_vs_openSMILE"] = np.where(comp["delta_rmse_vs_openSMILE"] < 0, "melhorou", np.where(comp["delta_rmse_vs_openSMILE"] > 0, "piorou", "empatou"))
    return comp.sort_values(["target", "delta_rmse_vs_openSMILE", "rmse_mean"]).reset_index(drop=True)


def model_factory(cfg: PipelineConfig, model_name: str):
    factories = {
        "DummyMean": lambda: (DummyRegressor(strategy="mean"), False),
        "Ridge": lambda: (Ridge(alpha=1.0, random_state=cfg.random_state), True),
        "LinearRegression": lambda: (LinearRegression(), True),
        "Lasso": lambda: (Lasso(alpha=0.001, random_state=cfg.random_state, max_iter=5000), True),
        "ElasticNet": lambda: (ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=cfg.random_state, max_iter=5000), True),
        "ExtraTrees": lambda: (ExtraTreesRegressor(n_estimators=cfg.n_estimators, min_samples_leaf=2, random_state=cfg.random_state, n_jobs=cfg.n_jobs_model), False),
        "RandomForest": lambda: (RandomForestRegressor(n_estimators=cfg.n_estimators, min_samples_leaf=2, random_state=cfg.random_state, n_jobs=cfg.n_jobs_model), False),
        "GradientBoosting": lambda: (GradientBoostingRegressor(n_estimators=cfg.n_estimators, random_state=cfg.random_state), False),
    }
    if model_name not in factories:
        model_name = "Ridge"
    return factories[model_name]()


def model_names_for_target(cfg: PipelineConfig, target: str, opensmile_best_by_target: pd.DataFrame) -> List[str]:
    names: List[str] = []
    all_models = ["Ridge", "LinearRegression", "Lasso", "ElasticNet", "ExtraTrees", "RandomForest", "GradientBoosting"]

    if cfg.model_mode == "all_models_panel":
        names.extend(all_models)
    elif cfg.model_mode == "compact_panel":
        names.extend(["Ridge", "ExtraTrees", "RandomForest", "GradientBoosting"])
    elif cfg.model_mode == "best_opensmile_per_target" and not opensmile_best_by_target.empty:
        match = opensmile_best_by_target[opensmile_best_by_target["target"].eq(target)]
        if not match.empty:
            names.append(str(match.iloc[0]["model"]))

    if cfg.include_ridge_reference:
        names.append("Ridge")
    if cfg.include_dummy_reference:
        names.append("DummyMean")

    return list(dict.fromkeys(names or ["Ridge"]))


def clean_xy(cfg: PipelineConfig, df_in: pd.DataFrame, features: List[str], target: str) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame]:
    existing = [f for f in dict.fromkeys(features) if f in df_in.columns]
    work_cols = existing + [target, cfg.song_id_col]
    work = df_in[work_cols].copy()
    work[target] = pd.to_numeric(work[target], errors="coerce")
    work = work.dropna(subset=[target, cfg.song_id_col]).reset_index(drop=True)
    X = work[existing].apply(pd.to_numeric, errors="coerce")
    X = X.drop(columns=X.columns[X.isna().all()], errors="ignore")
    X = X.drop(columns=X.columns[X.nunique(dropna=True) <= 1], errors="ignore")
    y = work[target]
    return X, y, work


def audit_feature_leakage(features: List[str]) -> None:
    leak_patterns = [r"song_id", r"block_idx", r"start_time", r"end_time", r"duration", r"valence", r"arousal", r"quadrant", r"title", r"artist", r"genre", r"target"]
    suspicious = []
    for feature in features:
        feature_lower = str(feature).lower()
        if any(re.search(pattern, feature_lower) for pattern in leak_patterns):
            suspicious.append(feature)
    if suspicious:
        raise ValueError(f"Features suspeitas de vazamento: {suspicious[:30]}")


def make_model_pipeline(model, needs_scaling: bool, selector_k: Optional[int], n_features: int) -> Pipeline:
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if selector_k is not None and n_features > 1:
        steps.append(("selector", SelectKBest(score_func=f_regression, k=min(selector_k, n_features))))
    if needs_scaling:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", clone(model)))
    return Pipeline(steps)


def evaluate_feature_set(cfg: PipelineConfig, df_in: pd.DataFrame, feature_set_name: str, features: List[str], target: str, model_name: str, selector_k: Optional[int] = None) -> Dict[str, object]:
    X, y, work = clean_xy(cfg, df_in, features, target)
    audit_feature_leakage(features)
    selector_name = "none" if selector_k is None else f"SelectKBest(k={selector_k})"
    if X.empty:
        return {"feature_set": feature_set_name, "target": target, "model": model_name, "selector": selector_name, "status": "no_valid_features", "n_samples": len(y), "n_features": 0}

    n_groups = work[cfg.song_id_col].nunique()
    if n_groups >= cfg.n_splits:
        cv = GroupKFold(n_splits=cfg.n_splits)
        split_kw = {"groups": work[cfg.song_id_col].values}
        cv_name = f"GroupKFold({cfg.n_splits}) by song"
    else:
        cv = KFold(n_splits=cfg.n_splits, shuffle=True, random_state=cfg.random_state)
        split_kw = {}
        cv_name = f"KFold({cfg.n_splits}) fallback"

    model, needs_scaling = model_factory(cfg, model_name)
    pipe = make_model_pipeline(model, needs_scaling=needs_scaling, selector_k=selector_k, n_features=X.shape[1])
    try:
        scores = cross_validate(pipe, X, y, cv=cv, scoring=SCORING, n_jobs=cfg.n_jobs_cv, error_score=np.nan, return_train_score=False, **split_kw)
        effective_k = X.shape[1] if selector_k is None else min(selector_k, X.shape[1])
        return {
            "feature_set": feature_set_name,
            "target": target,
            "model": model_name,
            "selector": "none" if selector_k is None else f"SelectKBest(k={effective_k})",
            "cv": cv_name,
            "n_samples": int(X.shape[0]),
            "n_songs": int(n_groups),
            "n_features": int(X.shape[1]),
            "rmse_mean": -float(np.nanmean(scores["test_rmse"])),
            "rmse_std": float(np.nanstd(-scores["test_rmse"])),
            "mae_mean": -float(np.nanmean(scores["test_mae"])),
            "mae_std": float(np.nanstd(-scores["test_mae"])),
            "r2_mean": float(np.nanmean(scores["test_r2"])),
            "r2_std": float(np.nanstd(scores["test_r2"])),
            "pearson_mean": float(np.nanmean(scores["test_pearson"])),
            "pearson_std": float(np.nanstd(scores["test_pearson"])),
            "status": "ok",
        }
    except Exception as exc:
        return {"feature_set": feature_set_name, "target": target, "model": model_name, "selector": selector_name, "cv": cv_name, "n_samples": int(X.shape[0]), "n_songs": int(n_groups), "n_features": int(X.shape[1]), "status": f"error: {exc}"}


def run_experiments(cfg: PipelineConfig, modeling_df: pd.DataFrame, feature_scenarios: Dict[str, List[str]], opensmile_best_by_target: pd.DataFrame) -> pd.DataFrame:
    all_rows = []
    if not cfg.run_fingerprint_experiments:
        return pd.DataFrame()
    for feature_set_name, features in progress_iter(feature_scenarios.items(), desc="Feature scenarios", total=len(feature_scenarios)):
        for target in cfg.targets:
            selector_options: List[Optional[int]] = [None]
            if cfg.run_selectkbest_inside_cv and len(features) > cfg.selector_k:
                selector_options.append(cfg.selector_k)
            for model_name in model_names_for_target(cfg, target, opensmile_best_by_target):
                for selector_k in selector_options:
                    all_rows.append(evaluate_feature_set(cfg, modeling_df, feature_set_name, features, target, model_name, selector_k=selector_k))
    results = pd.DataFrame(all_rows)
    if not results.empty:
        results = results.sort_values(["target", "rmse_mean", "feature_set", "model", "selector"], na_position="last").reset_index(drop=True)
    save_table(cfg, results, "fingerprint_vs_opensmile_feature_set_results.csv")
    return results


def best_result_tables(cfg: PipelineConfig, results: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if results.empty:
        best_by_feature_set = pd.DataFrame()
        best_new_by_target = pd.DataFrame()
    else:
        ok = results[results["status"].eq("ok")].copy()
        best_by_feature_set = ok.sort_values(["target", "feature_set", "rmse_mean"]).groupby(["target", "feature_set"], as_index=False).first()
        best_new_by_target = ok.sort_values(["target", "rmse_mean"]).groupby("target", as_index=False).first()
        dummy_by_target = ok[ok["model"].eq("DummyMean")].sort_values(["target", "rmse_mean"]).groupby("target", as_index=False).first()[["target", "rmse_mean"]].rename(columns={"rmse_mean": "dummy_rmse"}) if not ok[ok["model"].eq("DummyMean")].empty else pd.DataFrame(columns=["target", "dummy_rmse"])
        if not dummy_by_target.empty:
            best_by_feature_set = best_by_feature_set.merge(dummy_by_target, on="target", how="left")
            best_new_by_target = best_new_by_target.merge(dummy_by_target, on="target", how="left")
            best_by_feature_set["gain_vs_dummy_pct"] = 100.0 * (best_by_feature_set["dummy_rmse"] - best_by_feature_set["rmse_mean"]) / best_by_feature_set["dummy_rmse"]
            best_new_by_target["gain_vs_dummy_pct"] = 100.0 * (best_new_by_target["dummy_rmse"] - best_new_by_target["rmse_mean"]) / best_new_by_target["dummy_rmse"]
    save_table(cfg, best_by_feature_set, "best_result_by_target_and_feature_set.csv")
    save_table(cfg, best_new_by_target, "best_fingerprint_scenario_by_target.csv")
    return best_by_feature_set, best_new_by_target


def make_plots(cfg: PipelineConfig, best_by_feature_set: pd.DataFrame, comparison_vs_opensmile: pd.DataFrame) -> Dict[str, object]:
    figures: Dict[str, object] = {}
    if not best_by_feature_set.empty:
        fig_rmse = px.bar(best_by_feature_set, x="feature_set", y="rmse_mean", color="target", barmode="group", text="rmse_mean", hover_data=["model", "selector", "n_features", "r2_mean", "pearson_mean", "cv"], title="Melhor RMSE por conjunto de features (GroupKFold por musica)")
        fig_rmse.update_traces(texttemplate="%{text:.4f}", textposition="outside")
        fig_rmse.update_layout(template="plotly_white", xaxis_title="", yaxis_title="RMSE medio")
        save_plot(cfg, fig_rmse, "best_rmse_by_feature_set")
        if cfg.show_figures:
            fig_rmse.show()
        figures["best_rmse_by_feature_set"] = fig_rmse
    if not comparison_vs_opensmile.empty:
        plot_delta = comparison_vs_opensmile.copy()
        plot_delta["cenario"] = plot_delta["feature_set"] + " | " + plot_delta["model"] + " | " + plot_delta["selector"]
        fig_delta = px.bar(plot_delta, x="cenario", y="delta_rmse_vs_openSMILE", color="target", barmode="group", text="delta_rmse_vs_openSMILE", hover_data=["rmse_mean", "opensmile_reference_rmse", "impacto_vs_openSMILE", "delta_rmse_pct_vs_openSMILE"], title="Delta de RMSE contra o melhor baseline openSMILE salvo")
        fig_delta.add_hline(y=0, line_dash="dash", line_color="black")
        fig_delta.update_traces(texttemplate="%{text:.4f}", textposition="outside")
        fig_delta.update_layout(template="plotly_white", xaxis_title="", yaxis_title="RMSE novo - RMSE openSMILE")
        save_plot(cfg, fig_delta, "delta_rmse_vs_cached_opensmile")
        if cfg.show_figures:
            fig_delta.show()
        figures["delta_rmse_vs_cached_opensmile"] = fig_delta
    return figures


def build_tcc_summary(cfg: PipelineConfig, opensmile_best_by_target: pd.DataFrame, best_new_by_target: pd.DataFrame, comparison_vs_opensmile: pd.DataFrame, opensmile_block_cache: pd.DataFrame) -> str:
    fingerprint_only = not cfg.use_cached_opensmile_results and not cfg.rebuild_opensmile_block_cache
    if fingerprint_only:
        lines = ["Pipeline em modo fingerprint-only: openSMILE foi pulado nesta rodada."]
    else:
        lines = ["Pipeline refatorado em modo cache-first: openSMILE permanece opcional e so entra quando habilitado na configuracao."]

    lines.append("Os cenarios avaliados usam apenas os conjuntos de features de fingerprint disponiveis no inventario.")

    if cfg.use_cached_opensmile_results and not opensmile_best_by_target.empty:
        for _, row in opensmile_best_by_target.iterrows():
            lines.append(
                f"Baseline openSMILE salvo para {row['target']}: {row['model']} "
                f"com RMSE={row['rmse_mean']:.4f}, R2={row['r2_mean']:.4f} e {int(row['n_features'])} features."
            )
    elif cfg.use_cached_opensmile_results:
        lines.append("Baseline openSMILE salvo nao foi encontrado para comparacao direta.")
    else:
        lines.append("Baseline e diagnosticos openSMILE nao foram carregados porque use_cached_opensmile_results=False.")

    comparison_cols = {"target", "feature_set", "model", "selector", "delta_rmse_vs_openSMILE", "impacto_vs_openSMILE"}
    has_opensmile_comparison = not comparison_vs_opensmile.empty and comparison_cols.issubset(comparison_vs_opensmile.columns)

    if best_new_by_target.empty:
        lines.append("Nenhum resultado novo de fingerprint foi gerado nesta execucao.")
    else:
        for _, row in best_new_by_target.iterrows():
            metric_parts = [f"RMSE={row['rmse_mean']:.4f}"]
            if "r2_mean" in row.index and pd.notna(row.get("r2_mean")):
                metric_parts.append(f"R2={row['r2_mean']:.4f}")
            if "gain_vs_dummy_pct" in row.index and pd.notna(row.get("gain_vs_dummy_pct")):
                metric_parts.append(f"ganho_vs_dummy={row['gain_vs_dummy_pct']:.1f}%")

            base_line = (
                f"Melhor cenario de fingerprint para {row['target']}: {row['feature_set']} "
                f"com {row['model']} ({row['selector']}), " + ", ".join(metric_parts)
            )

            if has_opensmile_comparison:
                comp = comparison_vs_opensmile[
                    (comparison_vs_opensmile["target"].eq(row["target"]))
                    & (comparison_vs_opensmile["feature_set"].eq(row["feature_set"]))
                    & (comparison_vs_opensmile["model"].eq(row["model"]))
                    & (comparison_vs_opensmile["selector"].eq(row["selector"]))
                ]
                if not comp.empty and pd.notna(comp.iloc[0].get("delta_rmse_vs_openSMILE")):
                    delta = comp.iloc[0]["delta_rmse_vs_openSMILE"]
                    impact = comp.iloc[0]["impacto_vs_openSMILE"]
                    base_line += f"; isso {impact} em {abs(delta):.4f} RMSE contra o baseline openSMILE salvo"

            lines.append(base_line + ".")

    if fingerprint_only:
        lines.append("Cache openSMILE por bloco nao foi carregado nem reconstruido nesta rodada.")
    elif opensmile_block_cache.empty:
        lines.append("O cache openSMILE por bloco nao foi usado nesta comparacao.")
    else:
        lines.append("O cache openSMILE por bloco foi carregado, mas os cenarios de fingerprint continuam separados no inventario.")

    summary_text = "\n".join(lines)
    summary_path = cfg.tables_dir / "fingerprint_vs_opensmile_summary_for_tcc.txt"
    summary_path.write_text(summary_text, encoding="utf-8")
    print(summary_text)
    print(f"\nSaved summary: {summary_path}")
    return summary_text


## 11. openSMILE detalhado (pulado nesta rodada)

Esta etapa so roda quando `use_cached_opensmile_results=True`. No modo atual, o notebook segue apenas com fingerprints.


In [15]:
from IPython.display import Markdown, display

if cfg.use_cached_opensmile_results:
    opensmile_detail = load_detailed_opensmile_outputs(cfg)
    detailed_comparison = compare_fingerprint_results_with_detailed_opensmile(
        cfg,
        artifacts["fingerprint_results"],
        opensmile_detail,
    )
    detailed_figures = make_detailed_opensmile_comparison_plots(cfg, opensmile_detail, detailed_comparison)

    artifacts["opensmile_detail"] = opensmile_detail
    artifacts["detailed_comparison"] = detailed_comparison
    artifacts["detailed_figures"] = detailed_figures

    display(Markdown("### Manifesto de tabelas openSMILE carregadas"))
    display(opensmile_detail["manifest"])

    display(Markdown("### Referencias openSMILE padronizadas (long format)"))
    display(opensmile_detail["metric_reference_long"])

    display(Markdown("### Melhor referencia openSMILE por familia"))
    display(opensmile_detail["best_by_family"])

    display(Markdown("### Melhor fingerprint por conjunto vs melhor openSMILE por familia"))
    display(detailed_comparison["best_feature_set_vs_best_family"])

    display(Markdown("### Diagnostico openSMILE: features selecionadas por categoria"))
    display(opensmile_detail["selected_by_category"])

    display(Markdown("### Diagnostico openSMILE: top correlacoes por target"))
    display(opensmile_detail["top_correlations"])

    display(Markdown("### Diagnostico openSMILE: resumo EDA compacto"))
    display(opensmile_detail["eda_compact_summary"])
else:
    display(Markdown("### openSMILE detalhado pulado"))
    display(Markdown("Execucao em modo fingerprint-only (`use_cached_opensmile_results=False`)."))


### openSMILE detalhado pulado

Execucao em modo fingerprint-only (`use_cached_opensmile_results=False`).

## 12. Tabelas principais


In [16]:
from IPython.display import Markdown, display


def show_section(title: str, obj) -> None:
    if isinstance(obj, pd.DataFrame) and obj.empty:
        return
    display(Markdown(f"### {title}"))
    display(obj)


show_section("Inventario dos cenarios de *features*", artifacts["scenario_inventory"])
show_section("Resultados novos - *fingerprints*", artifacts["fingerprint_results"])
show_section("Melhor cenario de fingerprint por *target*", artifacts["best_new_by_target"])

show_section("Melhor baseline openSMILE salvo por *target*", artifacts.get("opensmile_best_by_target", pd.DataFrame()))
show_section("Delta em relacao ao baseline openSMILE salvo", artifacts.get("comparison_vs_opensmile", pd.DataFrame()))

if "opensmile_detail" in artifacts:
    show_section("openSMILE detalhado - melhor referencia por familia", artifacts["opensmile_detail"]["best_by_family"])

if "detailed_comparison" in artifacts:
    show_section(
        "Comparacao detalhada - *fingerprints* vs. familias openSMILE",
        artifacts["detailed_comparison"]["best_feature_set_vs_best_family"],
    )

if "fingerprint_correlations" in artifacts:
    corr_artifacts = artifacts["fingerprint_correlations"]
    family_titles = _FAMILY_LABELS
    for target_label in ["valence", "arousal"]:
        reports_by_family = corr_artifacts["reports"].get(target_label, {})
        if not reports_by_family:
            continue
        display(Markdown(f"### Correlacoes dos *fingerprints* - {target_label.capitalize()}"))
        for family_key, corr_df in reports_by_family.items():
            family_title = family_titles.get(family_key, family_key)
            display(Markdown(f"#### {family_title}"))
            display(corr_df.head(25))

    show_section(
        "Resumo das correlacoes dos *fingerprints* por categoria",
        corr_artifacts["summary_by_category"],
    )


### Inventario dos cenarios de *features*

,feature_set,scenario_family,source_type,n_features,n_samples,n_songs,uses_opensmile_block_cache,excluded_band,metric_group,band_group
0,Band Rank (bloco),fingerprint_source,block,252,6461,1802,False,,,
1,Rank (bloco),fingerprint_source,block,177,6461,1802,False,,,
2,Band Rank (raw peaks),fingerprint_source,raw_peaks,12,6461,1802,False,,,
3,Rank (raw peaks),fingerprint_source,raw_peaks,7,6461,1802,False,,,


### Resultados novos - *fingerprints*

,feature_set,target,model,selector,cv,n_samples,n_songs,n_features_requested,n_features_effective,n_features,rmse_mean,rmse_std,mae_mean,mae_std,r2_mean,r2_std,pearson_mean,pearson_std,status
0,Band Rank (bloco),arousal,GradientBoosting,none,GroupKFold(5) by song,6461,1802,252,252,252,0.188810,0.004183,0.151131,0.003442,0.543681,0.038340,7.388321e-01,2.594320e-02,ok
1,Band Rank (bloco),arousal,RandomForest,none,GroupKFold(5) by song,6461,1802,252,252,252,0.190441,0.004613,0.152984,0.003580,0.535635,0.040800,7.333863e-01,2.788637e-02,ok
2,Band Rank (bloco),arousal,GradientBoosting,SelectKBest(k=60),GroupKFold(5) by song,6461,1802,252,252,252,0.190590,0.004112,0.152366,0.003034,0.534865,0.040708,7.332290e-01,2.720566e-02,ok
3,Band Rank (bloco),arousal,ExtraTrees,none,GroupKFold(5) by song,6461,1802,252,252,252,0.190872,0.004983,0.153142,0.004327,0.533061,0.046128,7.318229e-01,3.258648e-02,ok
4,Band Rank (bloco),arousal,RandomForest,SelectKBest(k=60),GroupKFold(5) by song,6461,1802,252,252,252,0.191605,0.003901,0.153813,0.002822,0.530046,0.039221,7.302717e-01,2.569550e-02,ok
5,Band Rank (bloco),arousal,ExtraTrees,SelectKBest(k=60),GroupKFold(5) by song,6461,1802,252,252,252,0.191799,0.004338,0.153242,0.003541,0.528700,0.044020,7.291090e-01,2.975478e-02,ok
6,Band Rank (bloco),arousal,Ridge,SelectKBest(k=60),GroupKFold(5) by song,6461,1802,252,252,252,0.193493,0.002974,0.154642,0.002302,0.521512,0.027923,7.235322e-01,1.794484e-02,ok
7,Band Rank (bloco),arousal,Ridge,none,GroupKFold(5) by song,6461,1802,252,252,252,0.194886,0.005625,0.155016,0.004317,0.514734,0.031311,7.199355e-01,1.990393e-02,ok
8,Rank (bloco),arousal,GradientBoosting,none,GroupKFold(5) by song,6461,1802,177,177,177,0.195288,0.004610,0.155755,0.003300,0.511560,0.044415,7.165499e-01,3.199283e-02,ok
9,Rank (bloco),arousal,GradientBoosting,SelectKBest(k=60),GroupKFold(5) by song,6461,1802,177,177,177,0.196523,0.003989,0.156992,0.002652,0.505700,0.040130,7.125268e-01,2.872520e-02,ok


### Melhor cenario de fingerprint por *target*

...

,feature,feature_label,fingerprint_family,fingerprint_family_key,categoria_fingerprint,metric_group,target,pearson,abs_pearson,spearman,abs_spearman,n_valid
0,rawpeaks_band__magnitude_norm,raw_band::magnitude_norm,Band Rank (raw peaks),rawpeaks_band__,global,magnitude,arousal,0.368821,0.368821,0.360634,0.360634,6461
1,rawpeaks_band__pitch_class,raw_band::pitch_class,Band Rank (raw peaks),rawpeaks_band__,global,pitch_octave,arousal,-0.056310,0.056310,-0.054477,0.054477,6461
2,rawpeaks_band__frequency_hz,raw_band::frequency_hz,Band Rank (raw peaks),rawpeaks_band__,global,frequency,arousal,-0.049529,0.049529,-0.025369,0.025369,6461
3,rawpeaks_band__freq_bin,raw_band::freq_bin,Band Rank (raw peaks),rawpeaks_band__,global,frequency,arousal,-0.049529,0.049529,-0.025369,0.025369,6461
4,rawpeaks_band__peak_time_sec,raw_band::peak_time_sec,Band Rank (raw peaks),rawpeaks_band__,counts,counts,arousal,-0.039714,0.039714,-0.035034,0.035034,6461
5,rawpeaks_band__octave,raw_band::octave,Band Rank (raw peaks),rawpeaks_band__,global,pitch_octave,arousal,0.037471,0.037471,0.035466,0.035466,6461
6,rawpeaks_band__semitone,raw_band::semitone,Band Rank (raw peaks),rawpeaks_band__,global,pitch_octave,arousal,0.027888,0.027888,0.028836,0.028836,6461
7,rawpeaks_band__midi_note,raw_band::midi_note,Band Rank (raw peaks),rawpeaks_band__,global,pitch_octave,arousal,0.025988,0.025988,0.026521,0.026521,6461
8,rawpeaks_band__peak_time_relative_sec,raw_band::peak_time_relative_sec,Band Rank (raw peaks),rawpeaks_band__,counts,counts,arousal,-0.022655,0.022655,-0.027447,0.027447,6461
9,rawpeaks_band__frame_idx,raw_band::frame_idx,Band Rank (raw peaks),rawpeaks_band__,global,other,arousal,-0.022655,0.022655,-0.027447,0.027447,6461


#### Rank (raw peaks)

,feature,feature_label,fingerprint_family,fingerprint_family_key,categoria_fingerprint,metric_group,target,pearson,abs_pearson,spearman,abs_spearman,n_valid
0,rawpeaks_rank__midi_note,raw_rank::midi_note,Rank (raw peaks),rawpeaks_rank__,global,pitch_octave,arousal,-0.238878,0.238878,-0.240525,0.240525,6461
1,rawpeaks_rank__semitone,raw_rank::semitone,Rank (raw peaks),rawpeaks_rank__,global,pitch_octave,arousal,-0.238664,0.238664,-0.240643,0.240643,6461
2,rawpeaks_rank__octave,raw_rank::octave,Rank (raw peaks),rawpeaks_rank__,global,pitch_octave,arousal,-0.235562,0.235562,-0.237063,0.237063,6461
3,rawpeaks_rank__frequency_hz,raw_rank::frequency_hz,Rank (raw peaks),rawpeaks_rank__,global,frequency,arousal,-0.124497,0.124497,-0.216685,0.216685,6461
4,rawpeaks_rank__pitch_class,raw_rank::pitch_class,Rank (raw peaks),rawpeaks_rank__,global,pitch_octave,arousal,-0.112415,0.112415,-0.110021,0.110021,6461
5,rawpeaks_rank__peak_time_sec,raw_rank::peak_time_sec,Rank (raw peaks),rawpeaks_rank__,counts,counts,arousal,-0.039535,0.039535,-0.033538,0.033538,6461
6,rawpeaks_rank__peak_rank_global,raw_rank::peak_rank_global,Rank (raw peaks),rawpeaks_rank__,counts,rank,arousal,NaN,NaN,NaN,NaN,6461


### Resumo das correlacoes dos *fingerprints* por categoria

,target,fingerprint_family,fingerprint_family_key,categoria_fingerprint,metric_group,n_features,mean_abs_pearson,max_abs_pearson,mean_abs_spearman,max_abs_spearman
4,arousal,Band Rank (bloco),fpband__,high,other,3,0.565967,0.695876,0.639109,0.703429
16,arousal,Band Rank (bloco),fpband__,very_high,other,3,0.488608,0.608262,0.575844,0.628013
12,arousal,Band Rank (bloco),fpband__,mid,other,3,0.335411,0.539504,0.315463,0.545219
3,arousal,Band Rank (bloco),fpband__,high,magnitude,12,0.394488,0.465696,0.503161,0.567825
7,arousal,Band Rank (bloco),fpband__,low,magnitude,12,0.355085,0.439006,0.368602,0.459373
...,...,...,...,...,...,...,...,...,...,...
61,valence,Rank (bloco),fprank__,counts,pitch_octave,2,0.005086,0.005086,0.027861,0.027861
68,valence,Rank (raw peaks),rawpeaks_rank__,counts,counts,1,0.166312,0.166312,0.099781,0.099781
71,valence,Rank (raw peaks),rawpeaks_rank__,global,pitch_octave,4,0.144132,0.155212,0.158737,0.173371
70,valence,Rank (raw peaks),rawpeaks_rank__,global,frequency,1,0.075480,0.075480,0.160772,0.160772
